Queremos segmentar automáticamente las vértebras cervicais (C2–C4) en exames de videofluoroscopia de deglutição, para obter um escalar anatômico robusto. Hoje, essa anotação é manual, feita por especialistas (como a Raiane, no INCA), frame a frame no ImageJ, o que é demorado e sujeito a variabilidade inter-observador

In [ ]:
#CELDA0
!pip -q install --no-cache-dir --force-reinstall numpy==1.26.4


In [ ]:
print("holaSSSSaaFAA2FALaaasaaaasssAsds1212asasaasssasaVVabcsaasTssA Ccc")

In [ ]:
# ============================================================
# ZONA DE CONFIGURACIÓN DE ESCENARIOS  (MODIFICAR SOLO AQUÍ)
# ============================================================

SCENARIO_NAME = "3a_unet_semi_no_temp_CLAHE_CORREGIDO"  # etiqueta opcional para identificar el run
DEBUG = False

# Dataset
DATASET = "inca"  # "inca" o "corrosion"

# Lossname
loss_name = "bce_dice"  # "bce_dice_morpho","bce_dice_morpho_warmup", bce_dice_boundary_warmup       #bce_dice      bce_dice_hd_warmupwarm

# Arquitectura supervisada
ARCH      = "unetpp"          # "unet", "unetpp", "fpn", "pspnet", "deeplabv3+"
BACKBONE  = "efficientnet-b3" # encoder de segmentation_models_pytorch
N_CLASSES = 1                 # binaria

# Activar o no semi-supervisión (teacher–student)
USE_SEMI = True               # False = sólo supervisado, True = semi-superv.

# Preprocesamiento de imagen y suavizado de máscara
IMAGE_PREPROC  = "base"       # "base" o "denoise" "he" o "clahe_soft" "ad"
MASK_SMOOTHING = "none"       # "none", "morph", "gaussian"
USE_FIXED_CROP = False

# Hiperparámetros de semi-supervisión (teacher–student)
LAMBDA_U          = 0.05 #0.05      # peso de la loss no supervisada
TAU               = 0.95      # umbral de confianza para pseudo-labels
EMA_DECAY         = 0.99      # para el teacher (EMA)
SEMI_START_EPOCH  = 30        # épocas sólo supervisadas antes de activar semi
SEMI_WARMUP_EPOCHS = 20       # ramp up de LAMBDA_U

# Consistencia temporal (sólo si USE_SEMI=True y LAMBDA_T>0)
LAMBDA_T         = 0.00 #0.003      # peso de la loss temporal
MAX_TEMP_DELTA   = 2          # frames de separación máximo entre vecinos
TEMP_START_EPOCH = 4000         # a partir de qué época activar consistencia temp.
TEMP_WARMUP_EPOCHS = 5        # ramp up de LAMBDA_T
TAU_TEMP         = 0.7        # umbral de confianza para p0 en la loss temporal


In [ ]:
# === CELDA 0: CONFIGURACIÓN DE ENTORNO Y DATASET ===
from pathlib import Path
import os
import shutil
# ============================================================
# CONFIG DE MODO SOLO PARA DATASET = "inca"
# ============================================================
# OPCIONES:
#   "local" -> ejecutás en tu PC con la RTX 3060
#   "colab" -> ejecutás en Google Colab con GPU de Google (T4/V100/A100)
MODO = "colab"   # <--- solo aplica cuando DATASET == "inca"

# ============================================================
# SELECCIÓN DE DATASET
# ============================================================
if DATASET == "inca":
    # -----------------------------------------
    # RUTAS PARA INCA / UNM VÉRTEBRAS
    # -----------------------------------------
    if MODO == "local":
        # Ruta a tu dataset en tu PC (Google Drive sincronizado)
        DATA_ROOT = Path(r"G:\My Drive\UNM_vertebras_seg_v3")

    elif MODO == "colab":
        # Usás GPU de Google Colab
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_ROOT = Path("/content/drive/MyDrive/UNM_vertebras_seg_v3")

    else:
        raise ValueError("MODO debe ser 'local' o 'colab'.")

    # Comprobaciones básicas
    assert DATA_ROOT.exists(), f"No encuentro el dataset INCA en {DATA_ROOT}"

    train_images_dir = DATA_ROOT / "train" / "images"
    train_masks_dir  = DATA_ROOT / "train" / "masks"
    val_images_dir   = DATA_ROOT / "val" / "images"
    val_masks_dir    = DATA_ROOT / "val" / "masks"
    test_images_dir  = DATA_ROOT / "test" / "images"
    test_masks_dir   = DATA_ROOT / "test" / "masks"

    IMG_ROOT = str(DATA_ROOT)
    MSK_ROOT = str(DATA_ROOT)

    print("=== DATASET: INCA ===")
    print("MODO:", MODO)
    print("DATA_ROOT:", DATA_ROOT)
    print("Train imgs:", len(list(train_images_dir.iterdir())))
    print("Val imgs:",   len(list(val_images_dir.iterdir())))
    print("Test imgs:",  len(list(test_images_dir.iterdir())))

elif DATASET == "corrosion":
    # ----------------------------------------
    # === CELDA1 Cargar dataset preprocesado desde Drive (robusto a nombres de carpeta) ===
    import os, csv, zipfile, random, cv2, numpy as np, matplotlib.pyplot as plt
    from pathlib import Path

    # 1) Config
    GDRIVE_ID = "1WXxIc7VImXkKlDEG1G6M00d4h2DkXwf_"   # <- tu ZIP
    ZIP_PATH  = Path("/content/ameli_preproc.zip")
    EXTRACT_ROOT = Path("/content/_ameli_unzip_tmp")  # tmp de extracción

    # 2) Descargar con gdown
    !pip -q install gdown
    import gdown
    if not ZIP_PATH.exists():
        gdown.download(id=GDRIVE_ID, output=str(ZIP_PATH), quiet=False)
    else:
        print("✓ ZIP ya presente:", ZIP_PATH)

    # 3) Descomprimir a tmp limpio
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)

        # no borrar por seguridad; si quieres, haz: !rm -rf /content/_ameli_unzip_tmp
        pass
    else:
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    print("Descomprimiendo en", EXTRACT_ROOT)
    with zipfile.ZipFile(str(ZIP_PATH), "r") as zf:
        zf.extractall(str(EXTRACT_ROOT))

    # 4) Buscar carpeta "raíz del dataset" hasta 3 niveles
    def looks_like_split_root(p: Path) -> bool:
        return all((p/s).exists() for s in ["train","val","test"]) and \
              all(((p/s/"images").exists() and (p/s/"masks").exists()) for s in ["train","val","test"])

    def looks_like_processed_root(p: Path) -> bool:
        return (p/"processed/images").exists() and (p/"processed/masks").exists()

    def find_dataset_root(base: Path, max_depth=3):
        lvl = [base]; seen=set()
        split_cands, proc_cands = [], []
        for d in range(max_depth+1):
            nxt=[]
            for q in lvl:
                if q in seen: continue
                seen.add(q)
                if not q.exists() or not q.is_dir(): continue
                # ignora basura de macOS
                if q.name in {".DS_Store","__MACOSX"}: continue
                # chequeos
                if looks_like_split_root(q): split_cands.append((q,d))
                if looks_like_processed_root(q): proc_cands.append((q,d))
                # descender
                if d<max_depth:
                    for c in q.iterdir():
                        if c.is_dir() and not c.name.startswith("."):
                            nxt.append(c)
            lvl = nxt
        # preferimos split si existe; sino processed
        if split_cands:
            split_cands.sort(key=lambda x: x[1])
            return split_cands[0][0], "splits"
        if proc_cands:
            proc_cands.sort(key=lambda x: x[1])
            return proc_cands[0][0], "processed"
        return None, None

    DATA_ROOT, MODE = find_dataset_root(EXTRACT_ROOT, max_depth=3)
    assert DATA_ROOT is not None, f"No encontré una raíz con train/val/test o processed/ dentro de {EXTRACT_ROOT}"

    print(f"✓ Raíz detectada: {DATA_ROOT}")
    print(f"✓ Modo: {MODE}")

    # 5) Definir IMG_ROOT / MSK_ROOT para tu pipeline
    if MODE == "splits":
        IMG_ROOT = str(DATA_ROOT)   # {train,val,test}/images|masks
        MSK_ROOT = str(DATA_ROOT)
    else:
        IMG_ROOT = str(DATA_ROOT)   # processed/images|masks
        MSK_ROOT = str(DATA_ROOT)

    print("IMG_ROOT =", IMG_ROOT)
    print("MSK_ROOT =", MSK_ROOT)

    # 6) Mini verificación visual
    def pick_k_pairs(img_dir: Path, msk_dir: Path, k=4):
        names = sorted([p.stem for p in img_dir.glob("*.png") if (msk_dir/(p.stem + ".png")).exists()])
        if not names: return []
        random.seed(0)
        return random.sample(names, min(k, len(names)))

    def show_examples(img_dir: Path, msk_dir: Path, title="preview", k=4):
        picks = pick_k_pairs(img_dir, msk_dir, k)
        if not picks:
            print(f"sin pares en {img_dir}")
            return
        fig, axes = plt.subplots(len(picks), 3, figsize=(12, 3*len(picks)))
        if len(picks) == 1: axes = np.array([axes])
        for r, stem in enumerate(picks):
            ip = img_dir / f"{stem}.png"
            mp = msk_dir / f"{stem}.png"
            im = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
            mk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            overlay = im.copy(); overlay[mk > 0] = [255,0,0]
            axes[r,0].imshow(im);      axes[r,0].set_title(f"{title}/{stem}"); axes[r,0].axis("off")
            axes[r,1].imshow(mk, cmap="gray"); axes[r,1].set_title("mask bin (m>0)"); axes[r,1].axis("off")
            axes[r,2].imshow(overlay); axes[r,2].set_title("overlay"); axes[r,2].axis("off")
        plt.tight_layout(); plt.show()

    root = Path(IMG_ROOT)
    if MODE == "splits":
        for sp in ["train","val","test"]:
            imgd = root/sp/"images"
            mskd = root/sp/"masks"
            print(f"[{sp}] imgs={len(list(imgd.glob('*.png')))} | masks={len(list(mskd.glob('*.png')))}")
            if imgd.exists() and mskd.exists(): show_examples(imgd, mskd, title=sp, k=4)
    else:
        imgd = root/"processed/images"
        mskd = root/"processed/masks"
        print(f"[processed] imgs={len(list(imgd.glob('*.png')))} | masks={len(list(mskd.glob('*.png')))}")
        show_examples(imgd, mskd, title="processed", k=4)

    print("\nListo: usa IMG_ROOT/MSK_ROOT en el resto del notebook y SALTA la celda de preparación larga.")

    def init_epoch_components_logger(csv_path):
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        new_file = not os.path.exists(csv_path)
        f = open(csv_path, "a", newline="")
        w = csv.writer(f)
        if new_file:
            w.writerow([
                "epoch",
                "tr_L_sup", "tr_L_morph", "tr_lambda",
                "vl_L_sup", "vl_L_morph", "vl_lambda"
            ])
        return f, w
    # === RUTAS: usar dataset YA PREPROCESADO (con splits) ===
    from pathlib import Path

    DATA_ROOT = Path("/content/_ameli_unzip_tmp")  # <- lo que te detectó la celda de descarga
    assert DATA_ROOT.exists(), "No existe /content/_ameli_unzip_tmp, ejecuta la celda de descarga/descompresión."

    # Para TU pipeline (si usas IMG_ROOT/MSK_ROOT):
    IMG_ROOT = str(DATA_ROOT)
    MSK_ROOT = str(DATA_ROOT)

    # Si hay código viejo que esperaba estas variables, las definimos también para no romper nada:
    #AMELI_ROOT = DATA_ROOT
    #OUT_ROOT   = DATA_ROOT

    # Shape objetivo (tu dataset ya viene a 320x320)
    INPUT_SHAPE = (320, 320)

    print("Usando DATA_ROOT:", DATA_ROOT)
    print("IMG_ROOT =", IMG_ROOT)
    print("MSK_ROOT =", MSK_ROOT)



else:
    raise ValueError("DATASET debe ser 'inca' o 'corrosion'")


In [ ]:
!pip install scipy

In [ ]:
#CELDA 4
!pip -q install "torch==2.2.1+cu121" "torchvision==0.17.1+cu121" "torchaudio==2.2.1+cu121" -f https://download.pytorch.org/whl/torch_stable.html
#aaas
#aa

In [ ]:
##CELDA 5
# OpenCV headless 4.10 + Albumentations, SIN dependencias para no subir NumPy
!pip -q install opencv-python-headless==4.10.0.84 --no-deps
!pip -q install albumentations==1.3.1 --no-deps

# Resto de libs del proyecto
!pip -q install timm==0.9.2 segmentation-models-pytorch==0.3.3


In [ ]:
#CELDA3
import sys, numpy as np, torch, torchvision, torchaudio, cv2
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__, "| torchaudio:", torchaudio.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("opencv(headless):", cv2.__version__)


In [ ]:
#CELDA 6
import torch, numpy as np, cv2
print("CUDA ok?", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# mini-test tensor
x = torch.randn(2,3,256,256, device="cuda" if torch.cuda.is_available() else "cpu")
print("tensor sum:", float(x.sum()))


In [ ]:
!pip -q install qudida==0.0.4 --no-deps


In [ ]:
#CELDA 6b
# === [MORPH CONFIG] flags + bancos + builders de target ===
import json, numpy as np, cv2, torch, torch.nn.functional as F

# ----- FLAGS -----
MORPH_ON     = False                     # activar/desactivar la loss morfológica
MORPH_BANK   = "domain_w3"             # "rect3" | "domain_w2" | "domain_w3" | "dist_w2" | "dist_w3" | "dist_auto"
LAMBDA_MAX   = 0.2                      # peso máximo de la loss morfológica
WARMUP_EPOCHS_MORPH = 10                # warm-up de λ(t)
print(MORPH_BANK)
# ----- utilidades para kernels -----
def _clamp_int(v,a,b): return int(max(a, min(b, v)))

def _make_rect(h,w):
    return np.ones((h,w), np.uint8)

def _make_disk(r):
    y, x = np.ogrid[-r:r+1, -r:r+1]
    return ((x*x + y*y) <= r*r).astype(np.uint8)

def _make_line(angle_deg, length, width):
    S = int(max(length + 2*width + 8, 15))
    if S % 2 == 0: S += 1
    c = S//2
    y, x = np.mgrid[0:S, 0:S]
    x = x - c; y = y - c
    th = np.deg2rad(angle_deg % 180.0)
    A = np.array([-length/2.0, 0.0]); B = np.array([length/2.0, 0.0])
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    A = R @ A; B = R @ B; AB = B - A
    ab2 = (AB**2).sum()
    PX = x - A[0]; PY = y - A[1]
    t = np.clip((PX*AB[0] + PY*AB[1])/(ab2 + 1e-8), 0, 1)
    CX = A[0] + t*AB[0]; CY = A[1] + t*AB[1]
    dist = np.sqrt((x - CX)**2 + (y - CY)**2)
    return (dist <= (width/2.0)).astype(np.uint8)
###
def _cv_kernel_from_desc(desc):
    t,p = desc
    if t=="rect": return _make_rect(p["h"], p["w"])
    if t=="disk": return _make_disk(p["r"])
    if t=="line": return _make_line(p["angle"], p["length"], p["width"])
    raise ValueError(desc)
###
# ----- tol_px (ancho “sensible” del borde) -----
# Si ya corriste el análisis de dominio y guardaste domain_stats.json en tu drive, puedes leerlo;
# si no existe, usa un default robusto (5 px suele ir bien en AMELI).
try:
    with open("domain_stats.json", "r") as f:
        _stats = json.load(f)
        tol_px = int(np.clip(round(_stats.get("thickness_median", 5)/2), 2, 6))
except Exception:
    tol_px = 5

print("[MORPH] tol_px =", tol_px)

# ----- bancos DOM centrados en borde -----
def build_domain_bank(stats_or_none=None, W=2, tol=3, orient_top2=None):
    # Discos pequeños (multi-escala corta)
    r_small = max(2, int(round(tol)))
    r_med   = max(r_small+1, 3)
    r_big   = max(r_med+2, 5)
    bank = [("disk", {"r": int(r)}) for r in sorted({r_small, r_med, r_big})]
    # Líneas si hay orientaciones
    angs = []
    if stats_or_none and "orient_top2_deg" in stats_or_none:
        angs = stats_or_none["orient_top2_deg"] or []
    elif orient_top2:
        angs = orient_top2
    if len(angs) > 0:
        L = _clamp_int(24, 12, 48)
        for a in angs:
            bank.append(("line", {"angle": float(a), "length": L, "width": int(W)}))
    return bank

# Intenta leer orientaciones; si no, deja sólo discos
try:
    orient_top2 = _stats.get("orient_top2_deg", [])
except Exception:
    orient_top2 = []

BANK_BASE   = [("rect", {"h":3,"w":3})]
BANK_DOM_W2 = build_domain_bank(stats_or_none=locals().get("_stats", None), W=2, tol=tol_px, orient_top2=orient_top2)
BANK_DOM_W3 = build_domain_bank(stats_or_none=locals().get("_stats", None), W=3, tol=tol_px, orient_top2=orient_top2)

print("[MORPH] BANK_BASE  =", BANK_BASE)
print("[MORPH] BANK_DOM_W2=", BANK_DOM_W2)
print("[MORPH] BANK_DOM_W3=", BANK_DOM_W3)

# ----- builders de T (targets) -----
@torch.no_grad()
def build_target_kernels(yb: torch.Tensor, bank) -> torch.Tensor:
    """
    yb: [B,1,H,W] (0/1). Aplica cada kernel (dilate - erode) y promedia las coronas.
    Retorna T ∈ [0,1].
    """
    import numpy as np, cv2
    y_np = (yb.detach().cpu().numpy() > 0.5).astype(np.uint8)
    B,_,H,W = y_np.shape
    Ts = []
    for kdesc in bank:
        K = _cv_kernel_from_desc(kdesc)
        t_batch = np.zeros((B,1,H,W), np.uint8)
        for i in range(B):
            gt = y_np[i,0]
            dil = cv2.dilate(gt, K, 1)
            ero = cv2.erode (gt, K, 1)
            T  = ((dil - ero) > 0).astype(np.uint8)
            t_batch[i,0] = T
        Ts.append(t_batch)
    T_np = np.mean(np.stack(Ts, axis=0), axis=0)
    return torch.from_numpy(T_np).to(yb.device).float()

@torch.no_grad()
def build_target_distance(yb: torch.Tensor, w: int) -> torch.Tensor:
    """
    Banda por distancia robusta: gradiente morfológico + dilatación (ancho w px).
    yb: [B,1,H,W] (0/1). Retorna T {0,1} float.
    """
    import numpy as np, cv2
    y_np = (yb.detach().cpu().numpy() > 0.5).astype(np.uint8)
    B,_,H,W = y_np.shape
    T_np = np.zeros_like(y_np, dtype=np.uint8)
    se3 = np.ones((3,3), np.uint8)
    k   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*w+1, 2*w+1))
    for i in range(B):
        gt = y_np[i,0]
        edge = cv2.morphologyEx(gt, cv2.MORPH_GRADIENT, se3)
        band = cv2.dilate((edge*255).astype(np.uint8), k, 1)
        T_np[i,0] = (band>0).astype(np.uint8)
    return torch.from_numpy(T_np).to(yb.device).float()

# ----- loss morfológica -----
def morph_loss_from_logits(logits: torch.Tensor, T: torch.Tensor):
    """
    L_morph = 0.5*BCEwithLogits(p,T) + 0.5*(1 - Dice(p,T))
    """
    probs = torch.sigmoid(logits)
    bce_logits = F.binary_cross_entropy_with_logits(logits, T)
    eps = 1e-7
    inter = (probs*T).sum(dim=(1,2,3))
    denom = probs.sum(dim=(1,2,3)) + T.sum(dim=(1,2,3)) + eps
    dice_loss = 1.0 - (2*inter + eps)/denom
    return 0.5*bce_logits + 0.5*dice_loss.mean()

print("[MORPH] listo.")


In [ ]:
# #CELDA 12 [02] SETUP CONSOLIDADO — imports, seed, device, Drive, EXP_DIR
import os, cv2, time, random, platform
from typing import Dict, Tuple
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from PIL import Image

# ========= Reproducibilidad fuerte =========
SEED = 0  # <- cambia aquí si quieres otra semilla global
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# cuDNN determinista (evita heurísticas no deterministas)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# (opcional, más estricto; activa si quieres que PyTorch lance error ante ops no deterministas)
torch.use_deterministic_algorithms(False)

print(f"[SEED] Global = {SEED}")

# ========= Device info =========
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version(), "| Torch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cuda":
    try:
        print("GPU:", torch.cuda.get_device_name(0))
    except Exception as e:
        print("GPU info not available:", e)

# ========= Montar Drive solo si hace falta (Colab) =========
IN_COLAB = bool(os.environ.get("COLAB_GPU") or "COLAB_RELEASE_TAG" in os.environ)
if IN_COLAB:
    DRIVE_ROOT = "/content/drive"
    if not os.path.isdir(f"{DRIVE_ROOT}/MyDrive"):
        from google.colab import drive
        drive.mount(DRIVE_ROOT, force_remount=False)
    ROOT_SAVE_DIR = f"{DRIVE_ROOT}/MyDrive/corrosion_runs/efficient"
else:
    # Si no estás en Colab, guarda localmente en ./runs
    ROOT_SAVE_DIR = "./runs/corrosion/efficient"

os.makedirs(ROOT_SAVE_DIR, exist_ok=True)

# ========= MODO: NUEVO RUN o REANUDAR =========
RESUME_TIMESTAMP = None  # pon "YYYY-MM-DD-HH-MM" para reanudar; None = nuevo
LOSS_TAGbla = str(loss_name).replace("/", "-").replace(" ", "_")

TIMESTAMP = RESUME_TIMESTAMP or datetime.now().strftime("%Y-%m-%d-%H-%M")

EXP_DIR = os.path.join(ROOT_SAVE_DIR, f"{TIMESTAMP}_{LOSS_TAGbla}")

os.makedirs(EXP_DIR, exist_ok=True)

BEST_PATH = os.path.join(EXP_DIR, "model_best_val_loss.pth")
print("EXP_DIR:", EXP_DIR)
print("BEST_PATH existe:", os.path.isfile(BEST_PATH))

# (Opcional) carpetas locales auxiliares
# os.makedirs("checkpoints", exist_ok=True)
# os.makedirs("experiments", exist_ok=True)


In [ ]:
# ===== DEBUG FINGERPRINT HELPERS START =====
import json as _fp_json, platform as _fp_platform, \
       importlib.metadata as _fp_meta, time as _fp_time

def _fp_get_version(pkg):
    try: return _fp_meta.version(pkg)
    except Exception: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        _fp_json.dump({"pre_run": pre, "post_run": post}, f, indent=2, default=str)

def _fp_collect_pre_legacy():
    import sys
    # 1. Environment
    try: gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    except: gpu_name = "n/a"
    env = {
        "python": sys.version, "torch": torch.__version__,
        "torchvision": _fp_get_version("torchvision"),
        "cuda": torch.version.cuda,
        "cudnn": str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "n/a",
        "segmentation_models_pytorch": _fp_get_version("segmentation-models-pytorch"),
        "albumentations": _fp_get_version("albumentations"),
        "platform": _fp_platform.platform(), "gpu_name": gpu_name,
    }
    # 2. Code provenance (standalone notebook — no git hash)
    provenance = {
        "notebook_name": "r3_semiunet++_unlabeling_std_matched_r3_augALINEADOS__AUG_SUAVE_de_Copia_de_BASE_feb.ipynb",
        "git_hash": None,
    }
    # 3. Effective config — read from globals
    eff_cfg = {
        "SEED": SEED, "ARCH": ARCH, "BACKBONE": BACKBONE,
        "BATCH_SIZE": BATCH_SIZE, "NUM_AUGMENTED": NUM_AUGMENTED,
        "EPOCHS": EPOCHS, "LR": LR, "WARMUP_EPOCHS": WARMUP_EPOCHS,
        "PATIENCE_ES": PATIENCE_ES, "THRESH_EVAL": THRESH_EVAL,
        "USE_SEMI": USE_SEMI, "LAMBDA_U": LAMBDA_U, "TAU": TAU,
        "EMA_DECAY": EMA_DECAY, "SEMI_START_EPOCH": SEMI_START_EPOCH,
        "LAMBDA_T": LAMBDA_T, "INPUT_SHAPE": INPUT_SHAPE,
        "EXP_DIR": EXP_DIR,
        "unlabeled_batch_size": unlabeled_loader.batch_size if USE_SEMI else None,
    }
    # 4. Dataset / loader facts
    ds_facts = {
        "len_train_ds": len(train_ds), "len_val_ds": len(val_ds), "len_test_ds": len(test_ds),
        "len_unlabeled_ds": len(unlabeled_ds) if USE_SEMI else None,
        "train_loader_batch_size": train_loader.batch_size,
        "train_loader_num_workers": train_loader.num_workers,
        "train_loader_drop_last": train_loader.drop_last,
        "unlabeled_loader_batch_size": unlabeled_loader.batch_size if USE_SEMI else None,
        "unlabeled_loader_num_workers": unlabeled_loader.num_workers if USE_SEMI else None,
        "unlabeled_loader_drop_last": unlabeled_loader.drop_last if USE_SEMI else None,
    }
    # 5. Sample IDs — try known attribute names with fallback
    def _try_ids(ds):
        for attr in ("image_paths", "files", "img_paths", "samples"):
            v = getattr(ds, attr, None)
            if v is not None:
                return [str(x) for x in v[:5]]
        return "unavailable: no known attribute"
    sample_ids = {
        "first5_train": _try_ids(train_ds),
        "first5_unlabeled": _try_ids(unlabeled_ds) if USE_SEMI else None,
    }
    # 6. Batch shapes — analytical, no DataLoader consumed
    H, W = INPUT_SHAPE
    eff_bs = BATCH_SIZE * (1 + NUM_AUGMENTED)
    bs_u = max(1, BATCH_SIZE // 4)
    batch_shapes = {
        "xb": [eff_bs, 3, H, W], "yb": [eff_bs, 1, H, W],
        "xw_u": [bs_u, 3, H, W] if USE_SEMI else None,
        "xs_u": [bs_u, 3, H, W] if USE_SEMI else None,
        "note": "analytically derived — no DataLoader consumed",
    }
    # 7. Model fingerprint — model already instantiated, no RNG perturbation needed
    try:
        model_fp = {
            "total_params": sum(p.numel() for p in model.parameters()),
            "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "first_state_dict_keys": list(model.state_dict().keys())[:8],
        }
    except Exception as e:
        model_fp = {"error": str(e)}
    return {
        "timestamp_utc": _fp_time.strftime("%Y-%m-%dT%H:%M:%SZ", _fp_time.gmtime()),
        "environment": env, "provenance": provenance, "effective_cfg": eff_cfg,
        "dataset_facts": ds_facts, "sample_ids": sample_ids,
        "batch_shapes": batch_shapes, "model_fingerprint": model_fp,
    }

def _fp_collect_post_legacy(val_metrics, test_metrics):
    return {
        "timestamp_utc": _fp_time.strftime("%Y-%m-%dT%H:%M:%SZ", _fp_time.gmtime()),
        "best_path": BEST_PATH,
        "best_epoch_info": {
            "best_ckpt_score": best_ckpt_score,
            "best_val_loss": best_val_loss,
        },
        "val_metrics": {
            "f1_global": val_metrics.get("global_f1"),
            "iou_global": val_metrics.get("global_iou"),
            "f1_sample_mean": val_metrics.get("sample_mean_f1"),
            "iou_sample_mean": val_metrics.get("sample_mean_iou"),
        },
        "test_metrics": {
            "f1_global": test_metrics.get("global_f1"),
            "iou_global": test_metrics.get("global_iou"),
            "f1_sample_mean": test_metrics.get("sample_mean_f1"),
            "iou_sample_mean": test_metrics.get("sample_mean_iou"),
        },
        "finished_successfully": True,
        "exception": None,
    }

print("Debug fingerprint helpers loaded.")
# ===== DEBUG FINGERPRINT HELPERS END =====

In [ ]:
# #CELDA 13[LOG 0] Duplicar prints a pantalla y a archivo SIN tocar sys.stdout
import os, builtins
from datetime import datetime

LOSS_TAG = str(loss_name).replace("/", "-").replace(" ", "_")
RUN_LOG = os.path.join(EXP_DIR, f"log_{TIMESTAMP}_loss_{LOSS_TAG}.txt")

# Si ya estaba activo, restaurar y cerrar antes de reactivar
try:
    builtins.print = _orig_print  # restaura si existía
    try:
        _log_file.close()
    except Exception:
        pass
except NameError:
    pass

# Guardamos el print original y abrimos el archivo
_orig_print = builtins.print
_log_file = open(RUN_LOG, "w", encoding="utf-8")

def dual_print(*args, **kwargs):
    # 1) imprime normal en la celda
    _orig_print(*args, **kwargs)
    # 2) vuelca el mismo mensaje al archivo
    sep = kwargs.get("sep", " ")
    end = kwargs.get("end", "\n")
    text = sep.join(str(a) for a in args) + end
    _log_file.write(text)
    _log_file.flush()

builtins.print = dual_print  # <-- a partir de aquí, todo print duplica
print("=== RUN START ===", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("EXP_DIR:", EXP_DIR)
print("Log file:", RUN_LOG)
print("LOSS_NAME:", loss_name)

In [ ]:
print("EXP_DIR:", EXP_DIR)

In [ ]:
# ✅ CELDA 15 [04] PRECHECK — pares por split (versión robusta)
from pathlib import Path

# Verifica que IMG_ROOT/MSK_ROOT vienen de la celda de descarga/autodetección
try:
    IMG_ROOT, MSK_ROOT
except NameError:
    raise RuntimeError("Primero ejecuta la celda de rutas (donde se definen IMG_ROOT/MSK_ROOT).")

# Define los splits aquí
SPLITS = ["train", "val", "test"]

def stems(dirp: Path):
    p = Path(dirp)
    if not p.exists():
        return set()
    return {q.stem for q in p.glob("*.png")}

name_sets_img = {s: stems(Path(IMG_ROOT)/s/"images") for s in SPLITS}
name_sets_msk = {s: stems(Path(MSK_ROOT)/s/"masks")  for s in SPLITS}

res = {s: (len(name_sets_img[s]), len(name_sets_msk[s]), len(name_sets_img[s] & name_sets_msk[s])) for s in SPLITS}

print("Resumen (imgs, masks, pares por split):")
for s in SPLITS:
    ni, nm, npairs = res[s]
    print(f"  {s:>5}: imgs={ni:4d} | masks={nm:4d} | pares={npairs:4d}")

# Chequeo de posible fuga de nombres entre splits
for i, a in enumerate(SPLITS):
    for b in SPLITS[i+1:]:
        inter_img = name_sets_img[a] & name_sets_img[b]
        inter_msk = name_sets_msk[a] & name_sets_msk[b]
        if inter_img:
            print(f"⚠ fuga IMG: {len(inter_img)} coincidencias {a}↔{b} (ej: {sorted(list(inter_img))[:10]})")
        if inter_msk:
            print(f"⚠ fuga MSK: {len(inter_msk)} coincidencias {a}↔{b} (ej: {sorted(list(inter_msk))[:10]})")


In [ ]:
# CELDA 16] HYPERPARAMS — hiperparâmetros globais
INPUT_SHAPE: Tuple[int,int] = (320,320)
NUM_AUGMENTED = 5     # N vistas extras (train e val)
BATCH_SIZE    = 5
NUM_WORKERS   = 4
DROP_LAST     = True

LR            = 1e-3
WEIGHT_DECAY  = 1e-4
EPOCHS        = 2000      # limite alto; early stopping interrompe antes
WARMUP_EPOCHS = 10
PATIENCE_ES   = 20
THRESH_EVAL   = 0.5


In [ ]:
# #CELDA 17  PRECHECK — polaridade (diagnóstico)
import os, glob, numpy as np, cv2

def pos_ratios(split_dir, mode="white"):
    """
    Calcula la fracción de píxeles positivos en máscaras crudas 0/255.
    mode = "white" -> positivo = (mask > 0)
    mode = "black" -> positivo = (mask == 0)
    """
    mask_paths = sorted(glob.glob(os.path.join(split_dir, "*.png")) +
                        glob.glob(os.path.join(split_dir, "*.jpg")) +
                        glob.glob(os.path.join(split_dir, "*.jpeg")))
    vals = []
    for mp in mask_paths:
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if mode == "white":
            vals.append((m > 0).mean())
        else:
            vals.append((m == 0).mean())
    return np.array(vals, dtype=np.float32)

def _print_stats(name, arr):
    if arr.size == 0:
        print(f"{name}: (sin archivos)");
        return
    print(f"{name}: mean={arr.mean():.4f}  median={np.median(arr):.4f}  min={arr.min():.4f}  max={arr.max():.4f}  n={len(arr)}")

# Ajusta estas rutas a tus carpetas reales de máscaras crudas:
MSK_TRAIN = os.path.join(MSK_ROOT, "train", "masks")
MSK_VAL   = os.path.join(MSK_ROOT, "val",   "masks")
MSK_TEST  = os.path.join(MSK_ROOT, "test",  "masks")


print("\n[PRECHECK] Asumiendo positivo = BLANCO (mask>0)")
_print_stats("train", pos_ratios(MSK_TRAIN, "white"))
_print_stats("val  ", pos_ratios(MSK_VAL,   "white"))
_print_stats("test ", pos_ratios(MSK_TEST,  "white"))

print("\n[PRECHECK] Asumiendo positivo = NEGRO (mask==0)")
_print_stats("train", pos_ratios(MSK_TRAIN, "black"))
_print_stats("val  ", pos_ratios(MSK_VAL,   "black"))
_print_stats("test ", pos_ratios(MSK_TEST,  "black"))




In [ ]:
# #CELDA 18[07] PRECHECK razão  de positivos (m>0)
def positive_ratio_maior_que_zero(mask_path):
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    return (m>0).mean()

for split in SPLITS:
    vals=[]
    for f in (Path(MSK_ROOT)/split/"masks").glob("*.png"):
        r = positive_ratio_maior_que_zero(str(f))
        if r is not None: vals.append(r)
    if vals:
        arr = np.array(vals)
        print(f"{split}: pos_ratio(m>0) mean={arr.mean():.4f}  med={np.median(arr):.4f}  min={arr.min():.4f}  max={arr.max():.4f}")


In [ ]:
print(DEBUG)

In [ ]:
##CELDA 19 [08] PREPROCESS — pad→resize→[0,1] + binarização (ALINHADO AO RUNNER): 1 = corrosão (mask > 0 após inverter)

def smooth_mask(mask_uint8: np.ndarray, mode: str = "none") -> np.ndarray:
    mask = mask_uint8.astype(np.uint8)

    if mode == "none":
        return mask

    if mode == "gaussian":
        # Gaussian blur + binarización (Otsu)
        blurred = cv2.GaussianBlur(mask, (5, 5), 0)
        _, mask_bin = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return mask_bin

    if mode == "morph":
        kernel = np.ones((3, 3), np.uint8)
        opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
        closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=1)
        return closed

    return mask

def apply_image_preproc(image: np.ndarray, mode: str = "base") -> np.ndarray:
    """
    Preprocesamiento de IMAGEN.
    Espera RGB uint8 [H,W,3], devuelve RGB uint8 [H,W,3].
    """
    if mode == "base":
        return image

    # --- 1) Denoise only (preserva bordes, no cambia contraste local) ---
    if mode == "denoise":
        return cv2.bilateralFilter(image, d=5, sigmaColor=75, sigmaSpace=75)

    # --- 2) HE (histogram equalization global) en luminancia ---
    if mode == "he":
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l_eq = cv2.equalizeHist(l)
        lab_eq = cv2.merge((l_eq, a, b))
        return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)

    # --- 3) CLAHE (tu versión actual) ---
    if mode == "clahe":
        image_dn = cv2.bilateralFilter(image, d=5, sigmaColor=75, sigmaSpace=75)
        lab = cv2.cvtColor(image_dn, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        lab = cv2.merge((cl, a, b))
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # --- 4) CLAHE suave (menos agresivo) ---
    if mode == "clahe_soft":
        # sin denoise o con denoise leve: prueba las dos si quieres
        image_dn = cv2.bilateralFilter(image, d=5, sigmaColor=50, sigmaSpace=50)
        lab = cv2.cvtColor(image_dn, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        # parámetros más conservadores
        clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(16, 16))
        cl = clahe.apply(l)
        lab = cv2.merge((cl, a, b))
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # --- 5) AD (Anisotropic Diffusion) aproximado con Perona–Malik ---
    # Nota: OpenCV no trae AD directo; implementamos una versión simple.
    if mode == "ad":
        # convertir a gris float para difundir
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)

        # parámetros típicos (ajustables)
        n_iter = 10
        kappa = 30.0
        gamma = 0.15

        for _ in range(n_iter):
            # gradientes N,S,E,W
            nablaN = np.roll(gray, -1, axis=0) - gray
            nablaS = np.roll(gray,  1, axis=0) - gray
            nablaE = np.roll(gray, -1, axis=1) - gray
            nablaW = np.roll(gray,  1, axis=1) - gray

            # funciones de conducción (Perona-Malik g1)
            cN = np.exp(-(nablaN/kappa)**2)
            cS = np.exp(-(nablaS/kappa)**2)
            cE = np.exp(-(nablaE/kappa)**2)
            cW = np.exp(-(nablaW/kappa)**2)

            gray = gray + gamma*(cN*nablaN + cS*nablaS + cE*nablaE + cW*nablaW)

        # volver a uint8 y a 3 canales RGB
        gray = np.clip(gray, 0, 255).astype(np.uint8)
        rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        return rgb

    raise ValueError(f"modo de preprocesamiento de imagen no soportado: {mode}")


def pad_to_square(img: Image.Image, fill_color=(0, 0, 0)) -> Image.Image:
    w, h = img.size
    side = max(w, h)
    canvas = Image.new(img.mode, (side, side), color=fill_color)
    canvas.paste(img, ((side-w)//2, (side-h)//2))
    return canvas

def get_imagenet_preprocessing(target_size=(320,320), use_pad=True, imagenet_norm=False):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def preprocess(image_uint8: np.ndarray, mask_uint8: np.ndarray) -> Dict[str, np.ndarray]:
        # 1) Pad: para positivo=branco, o pad da máscara deve ser 0 (fundo)
        if use_pad:
            img_pil = Image.fromarray(image_uint8)
            image = np.array(pad_to_square(img_pil, fill_color=(0,0,0)))
            m = np.squeeze(mask_uint8)
            mask = np.array(pad_to_square(Image.fromarray(m), fill_color=0))
        else:
            image, mask = image_uint8, np.squeeze(mask_uint8)

        # 2) resize
        W,H = target_size
        image = cv2.resize(image, (W,H), interpolation=cv2.INTER_LINEAR)
        mask  = cv2.resize(mask,  (W,H), interpolation=cv2.INTER_NEAREST)

        # DEBUG: inspeccionar tipo y rango ANTES y DESPUÉS del preproc
        if DEBUG:
            print("\n[DEBUG] BEFORE apply_image_preproc")
            print("dtype:", image.dtype, "shape:", image.shape)
            print("min/max:", float(image.min()), float(image.max()))

        # 2.5) preprocesamiento de imagen (CLAHE, etc.)
        image = apply_image_preproc(image, IMAGE_PREPROC)

        if DEBUG:
            print("[DEBUG] AFTER apply_image_preproc")
            print("dtype:", image.dtype, "shape:", image.shape)
            print("min/max:", float(image.min()), float(image.max()))

        # SUAVIZAR MÁSCARA AQUÍ (opcional)
        if MASK_SMOOTHING != "none":
            mask = smooth_mask(mask, mode=MASK_SMOOTHING)


        # 3) imagem para [0,1]
        image = image.astype(np.float32)/255.0
        if imagenet_norm:
            image = (image-mean)/std

        # 4) INVERTER máscara crua (se o dataset original tem corrosão em preto)
        # mask = 255 - mask  # agora a corrosão vira branco

        # 5) binário alinhado ao runner: 1 = corrosão (mask > 0)
        mask = (mask > 0).astype(np.float32)

        if mask.ndim == 2:
            mask = mask[..., None]
        return {"image": image, "mask": mask}
    return preprocess

PREPROCESS = get_imagenet_preprocessing(target_size=INPUT_SHAPE, use_pad=True, imagenet_norm=False)
print(f"Preprocess pronto: {INPUT_SHAPE}, img∈[0,1], positivo=(mask>0) -> 1=corrosão (alinhado ao runner)")


In [ ]:
def PREPROCESS_IMAGE_ONLY(image_uint8: np.ndarray) -> torch.Tensor:
    img_pil = Image.fromarray(image_uint8.astype(np.uint8))
    image = np.array(pad_to_square(img_pil, fill_color=(0, 0, 0)))

    W, H = INPUT_SHAPE
    image = cv2.resize(image, (W, H), interpolation=cv2.INTER_LINEAR)

    image = apply_image_preproc(image, IMAGE_PREPROC)

    image = image.astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))  # [C,H,W]

    return torch.from_numpy(image).float()

In [ ]:
import matplotlib.pyplot as plt

train_masks_dir = os.path.join(MSK_ROOT, "train", "masks")
mask_files = [f for f in os.listdir(train_masks_dir) if f.endswith(".png")]
mask_path = os.path.join(train_masks_dir, mask_files[0])

raw = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

m_none = smooth_mask(raw, mode="none")
m_morph = smooth_mask(raw, mode="morph")
m_gauss = smooth_mask(raw, mode="gaussian")

fig, axs = plt.subplots(1, 4, figsize=(16,4))
axs[0].imshow(raw, cmap="gray");   axs[0].set_title("raw");       axs[0].axis("off")
axs[1].imshow(m_none, cmap="gray");axs[1].set_title("none");      axs[1].axis("off")
axs[2].imshow(m_morph, cmap="gray");axs[2].set_title("morph");    axs[2].axis("off")
axs[3].imshow(m_gauss, cmap="gray");axs[3].set_title("gaussian"); axs[3].axis("off")
plt.show()


In [ ]:
print("hola")

In [ ]:
# CELDA 20 [09] TRANSFORMS — Albumentations (train; val/test sin augs)
def build_transforms():
    train_tf = A.Compose([
        # Geometría MUY suave
        A.ShiftScaleRotate(
            shift_limit=0.01, scale_limit=0.03, rotate_limit=5,
            border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5
        ),

        # Flip horizontal muy raro (si no rompe dominio)
        #A.HorizontalFlip(p=0.1),

        # Fotométricas suaves (exposición/contraste)
        A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.4),
        A.RandomGamma(gamma_limit=(90, 110), p=0.2),
        #A.ShotNoise(scale_range=(0.1, 0.3), p=0.15),

        # Ruido muy leve A
        A.GaussNoise(var_limit=(3.0, 12.0), p=0.15),
        #A.CLAHE(clip_limit=(1, 2), tile_grid_size=(8, 8), p=0.10),
    ])
    val_tf  = None
    test_tf = None
    print("Augs: TRAIN=SUAVE | VAL=sin augs | TEST=sin augs.")
    return train_tf, val_tf, test_tf

# CELDA 20b — Transforms para rama débil / fuerte (semi-supervisado)
def build_semi_transforms(target_size=(320, 320)):
    weak = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.01, scale_limit=0.02, rotate_limit=3,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5
        ),
        A.HorizontalFlip(p=0.1),
    ])

    strong = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.015, scale_limit=0.04, rotate_limit=6,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6
        ),
        A.HorizontalFlip(p=0.15),

        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.6),
        A.RandomGamma(gamma_limit=(88, 112), p=0.3),
       # A.ShotNoise(scale_range=(0.1, 0.3), p=0.15),

        A.GaussNoise(var_limit=(4.0, 16.0), p=0.25),
        #A.CLAHE(clip_limit=(1, 2), tile_grid_size=(8, 8), p=0.10),
    ])
    return weak, strong

TRAIN_TF, VAL_TF, TEST_TF = build_transforms()
AUG_WEAK, AUG_STRONG = build_semi_transforms()


In [ ]:
# CELDA NUEVA — calcular CROP_BOX a partir de masks de train
import os
import cv2
import numpy as np
from glob import glob

train_mask_dir = os.path.join(MSK_ROOT, "train", "masks")
mask_paths = sorted(glob(os.path.join(train_mask_dir, "*.png")))

boxes = []

for p in mask_paths:
    m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if m is None:
        continue

    ys, xs = np.where(m > 0)
    if len(xs) == 0 or len(ys) == 0:
        continue

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    boxes.append([x1, y1, x2, y2])

boxes = np.array(boxes)
print("máscaras válidas:", len(boxes))

x1 = int(np.percentile(boxes[:, 0], 2))
y1 = int(np.percentile(boxes[:, 1], 2))
x2 = int(np.percentile(boxes[:, 2], 98))
y2 = int(np.percentile(boxes[:, 3], 98))

margin_x = 80
margin_y = 100

x1 = max(0, x1 - margin_x)
y1 = max(0, y1 - margin_y)
x2 = x2 + margin_x
y2 = y2 + margin_y

CROP_BOX = (x1, y1, x2, y2)
print("CROP_BOX =", CROP_BOX)

In [ ]:
def apply_fixed_crop(img, mask=None, crop_box=CROP_BOX):
    x1, y1, x2, y2 = crop_box
    img = img[y1:y2, x1:x2]

    if mask is not None:
        mask = mask[y1:y2, x1:x2]
        return img, mask

    return img

In [ ]:
# #CELDA 21 [10] DATASET [original + N augs] em uma única classe
class CorrosionDatasetMono(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, num_augmented=5):
        self.transform = transform
        self.n = int(max(0, num_augmented))
        imgs = {f for f in os.listdir(images_dir) if f.lower().endswith(".png")}
        msks = {f for f in os.listdir(masks_dir) if f.lower().endswith(".png")}
        common = sorted(list(imgs & msks))
        missing = imgs ^ msks
        if missing: print(f"⚠️ {len(missing)} sem par serão ignorados.")
        assert common, "sem pares .png imagem/máscara."
        self.pairs = [(os.path.join(images_dir, n), os.path.join(masks_dir, n)) for n in common]

    def __len__(self): return len(self.pairs)

    def _to_tensor_pair(self, image_uint8, mask_uint8):
        pp = PREPROCESS(image_uint8, mask_uint8)
        im, ms = pp["image"], pp["mask"]
        x = torch.from_numpy(im).permute(2,0,1).float()         # [3,H,W]
        y = torch.from_numpy(ms[...,0]).unsqueeze(0).float()    # [1,H,W]
        return x, y

    def __getitem__(self, idx):
        ip, mp = self.pairs[idx]
        image0 = cv2.imread(ip, cv2.IMREAD_COLOR)[:, :, ::-1]
        mask0  = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)

        if USE_FIXED_CROP:
            image0, mask0 = apply_fixed_crop(image0, mask0, CROP_BOX)

        views = [self._to_tensor_pair(image0, mask0)]  # original sem augs
        for _ in range(self.n):                        # N augs
            if self.transform is not None:
                out = self.transform(image=image0, mask=mask0)
                img_i, msk_i = out["image"], out["mask"]
            else:
                img_i, msk_i = image0, mask0
            views.append(self._to_tensor_pair(img_i, msk_i))
        return views

# CELDA 21b — Dataset para FRAMES NO ETIQUETADOS (UNM)
class UnlabeledFramesDataset(Dataset):
    def __init__(self, images_dir, transform_weak=None, transform_strong=None,
                 target_size=INPUT_SHAPE, use_pad=True):
        self.images_dir = str(images_dir)
        self.weak_tf = transform_weak
        self.strong_tf = transform_strong
        self.target_size = tuple(target_size)
        self.use_pad = bool(use_pad)

        self.files = sorted(
            f for f in os.listdir(self.images_dir)
            if f.lower().endswith(".png")
        )
        assert len(self.files) > 0, f"⚠️ No se encontraron PNG en {self.images_dir}"

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        path = os.path.join(self.images_dir, fname)

        # Leer RGB
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"No pude leer {path}")
        img = img[:, :, ::-1]  # BGR -> RGB

        # Dos copias de la imagen original
        weak_img = img.copy()
        strong_img = img.copy()

        # Augmentations sobre la imagen original
        if self.weak_tf is not None:
            weak_img = self.weak_tf(image=weak_img)["image"]
        if self.strong_tf is not None:
            strong_img = self.strong_tf(image=strong_img)["image"]

        # Luego preprocess base, igual que en supervisado
        weak_img = PREPROCESS_IMAGE_ONLY(weak_img)
        strong_img = PREPROCESS_IMAGE_ONLY(strong_img)

        return weak_img, strong_img

# Dataset para pares temporales de frames UNLABELED
class TemporalUnlabeledPairsDataset(Dataset):
    """
    Construye pares (frame_t, frame_{t+Δ}) del mismo video a partir de los PNG
    de unlabeling/images. Usa como máximo un desplazamiento temporal MAX_TEMP_DELTA.
    """
    def __init__(self, images_dir, transform=None,
                 target_size=INPUT_SHAPE, use_pad=True,
                 max_delta=MAX_TEMP_DELTA):
        self.images_dir = str(images_dir)
        self.transform = transform
        self.target_size = tuple(target_size)
        self.use_pad = bool(use_pad)
        self.max_delta = int(max_delta)

        # Lista de todos los PNG, ordenados
        all_files = sorted(
            f for f in os.listdir(self.images_dir)
            if f.lower().endswith(".png")
        )
        assert len(all_files) > 0, f"⚠️ No se encontraron PNG en {self.images_dir}"

        # Parsear nombres tipo v025_f239.png -> (vid=25, frame=239)
        def parse_name(fname):
            base = os.path.splitext(fname)[0]  # v025_f239
            vpart, fpart = base.split("_")     # "v025", "f239"
            vid = int(vpart[1:])
            frame = int(fpart[1:])
            return vid, frame

        parsed = [parse_name(f) for f in all_files]

        pairs = []
        # Recorremos y formamos pares dentro del mismo video
        for i in range(len(all_files) - 1):
            vid0, frame0 = parsed[i]
            # Miramos hacia delante mientras sea el mismo video
            j = i + 1
            while j < len(all_files):
                vid1, frame1 = parsed[j]
                if vid1 != vid0:
                    break  # cambiamos de video, salimos del while
                delta = frame1 - frame0
                if delta <= 0:
                    j += 1
                    continue
                if delta <= self.max_delta:
                    pairs.append((all_files[i], all_files[j]))
                    j += 1
                else:
                    # ya nos pasamos del rango temporal permitido
                    break

        assert len(pairs) > 0, f"⚠️ No se formaron pares temporales en {self.images_dir}"
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def _load_img(self, fname):
        path = os.path.join(self.images_dir, fname)
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"No se pudo leer {path}")

        img = img[:, :, ::-1]  # BGR -> RGB

        if self.transform is not None:
            out = self.transform(image=img)
            img = out["image"]

        img = PREPROCESS_IMAGE_ONLY(img)
        return img


    def __getitem__(self, idx):
        fname0, fname1 = self.pairs[idx]
        x0 = self._load_img(fname0)
        x1 = self._load_img(fname1)
        return x0, x1


In [ ]:
#CELDA 22# [11] COLLATE — flatten para achatar vistas em batch plano
def flatten_collate(batch):
    flat = [item for sublist in batch for item in sublist]
    xs, ys = zip(*flat)
    return torch.stack(xs, 0), torch.stack(ys, 0)


In [ ]:
#CELDA 23# [12] DATALOADERS — train con augs (N=NUM_AUGMENTED) | val/test sin augs (N=0)
import random, numpy as np, torch
from torch.utils.data import DataLoader

# --- siembra para DataLoader: cada worker hereda una semilla reproducible ---
g = torch.Generator()
g.manual_seed(SEED)  # SEED ya lo definiste en HPARAMS

def _seed_worker(worker_id):
    seed = SEED + worker_id
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# --- datasets ---
train_ds = CorrosionDatasetMono(
    os.path.join(IMG_ROOT,"train","images"),
    os.path.join(MSK_ROOT,"train","masks"),
    transform=TRAIN_TF, num_augmented=NUM_AUGMENTED)

# VAL/TEST sin augmentations para métricas estables
val_ds = CorrosionDatasetMono(
    os.path.join(IMG_ROOT,"val","images"),
    os.path.join(MSK_ROOT,"val","masks"),
    transform=None, num_augmented=0)

test_ds = CorrosionDatasetMono(
    os.path.join(IMG_ROOT,"test","images"),
    os.path.join(MSK_ROOT,"test","masks"),
    transform=None, num_augmented=0)

# --- dataloaders (deterministas) ---
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=DROP_LAST,
    collate_fn=flatten_collate,
    worker_init_fn=_seed_worker, generator=g,
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    collate_fn=flatten_collate,
    worker_init_fn=_seed_worker, generator=g,
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    collate_fn=flatten_collate,
    worker_init_fn=_seed_worker, generator=g,
)

print(f"lote efectivo TRAIN: {BATCH_SIZE} * (1+{NUM_AUGMENTED}) = {BATCH_SIZE*(1+NUM_AUGMENTED)}")
print("VAL/TEST: sin augmentations (1 vista por imagen) — DataLoaders sembrados para reproducibilidad")
#  DATALOADER UNLABELED (solo si USE_SEMI)
if USE_SEMI:
    WEAK_TF, STRONG_TF = build_semi_transforms(target_size=INPUT_SHAPE)

    unlabeled_images_dir = os.path.join(IMG_ROOT, "unlabeling_std_matched_r3", "images")
    if os.path.isdir(unlabeled_images_dir):
        unlabeled_ds = UnlabeledFramesDataset(
            images_dir=unlabeled_images_dir,
            transform_weak=WEAK_TF,
            transform_strong=STRONG_TF,
            target_size=INPUT_SHAPE
        )


        BATCH_SIZE_UNLAB = max(1, BATCH_SIZE // 4)  # o incluso //4 si hace falta (antes era //2)

        unlabeled_loader = DataLoader(
            unlabeled_ds,
            batch_size=BATCH_SIZE_UNLAB,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=True,
            worker_init_fn=_seed_worker,
            generator=g,
        )
        print("Unlabeled imgs:", len(unlabeled_ds))
        print("unlabeled_images_dir is :", unlabeled_images_dir)
        # NUEVO: dataloader para consistencia temporal
        temporal_unlab_ds = TemporalUnlabeledPairsDataset(
            images_dir=unlabeled_images_dir,
            transform=WEAK_TF,          # vistas suaves, sin augs fuertes
            target_size=INPUT_SHAPE,
            use_pad=True,
            max_delta=MAX_TEMP_DELTA,
        )

        TEMP_BATCH_SIZE = BATCH_SIZE_UNLAB  # por simplicidad igual
        temporal_unlab_loader = DataLoader(
            temporal_unlab_ds,
            batch_size=TEMP_BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=True,
            worker_init_fn=_seed_worker,
            generator=g,
        )
        print("Unlabeled pares temporales:", len(temporal_unlab_ds))

    else:
        print(f"[WARN] No existe carpeta de unlabeled en {unlabeled_images_dir}")
        unlabeled_loader = None
        temporal_unlab_loader = None
else:
    unlabeled_loader = None
    temporal_unlab_loader = None



In [ ]:
#CELDA 24
# [13] VISUAL CHECK — imagen, máscara cruda, binaria y overlay (antes de entrenar)
def show_examples_ds(images_root, masks_root, split="val", k=4):
    img_dir = os.path.join(images_root, split, "images")  # << cambia
    msk_dir = os.path.join(masks_root, split, "masks")    # << cambia
    imgs = {f for f in os.listdir(img_dir) if f.lower().endswith(".png")}
    msks = {f for f in os.listdir(msk_dir) if f.lower().endswith(".png")}
    names = sorted(list(imgs & msks))
    if not names:
        print("sem pares em", split)
        return
    picks = random.sample(names, min(k, len(names)))

    fig, axes = plt.subplots(len(picks), 4, figsize=(12, 3*len(picks)))
    if len(picks) == 1:
        axes = np.array([axes])

    for r, name in enumerate(picks):
        ip = os.path.join(img_dir, name)
        mp = os.path.join(msk_dir, name)
        img  = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        mraw = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)

        # binaria coherente con el preprocess “fondo blanco”:
        # invertimos crudo y luego usamos positivo = blanco (mask > 0)
        #mbin = ((255 - mraw) > 0)
        mbin = (mraw > 0)

        overlay = img.copy()
        overlay[mbin.astype(bool)] = [255, 0, 0]   # rojo sobre corrosión

        axes[r,0].imshow(img);    axes[r,0].set_title(f"{split}/{name}"); axes[r,0].axis("off")
        axes[r,1].imshow(mraw, cmap="gray"); axes[r,1].set_title("máscara cruda"); axes[r,1].axis("off")
        axes[r,2].imshow(mbin, cmap="gray"); axes[r,2].set_title("binaria: (mask>0)"); axes[r,2].axis("off")
        axes[r,3].imshow(overlay);           axes[r,3].set_title("overlay"); axes[r,3].axis("off")

    plt.tight_layout()
    plt.show()

show_examples_ds(IMG_ROOT, MSK_ROOT, split="val", k=4)


In [ ]:
#CELDA 25
# [14] MODELO — Unet EfficientNet-B3
def create_model(arch_name, backbone, n_classes):
    arch = arch_name.lower()
    if arch in ["unet", "u-net"]:
        return smp.Unet(encoder_name=backbone, encoder_weights="imagenet", in_channels=3, classes=n_classes)
    elif arch in ["deeplabv3+", "deeplabv3plus", "deeplabv3"]:
        return smp.DeepLabV3Plus(encoder_name=backbone, encoder_weights="imagenet", in_channels=3, classes=n_classes)
    elif arch == "unetpp":
        return smp.UnetPlusPlus(encoder_name=backbone, encoder_weights="imagenet", in_channels=3, classes=n_classes)
    elif arch == "fpn":
        return smp.FPN(encoder_name=backbone, encoder_weights="imagenet", in_channels=3, classes=n_classes)
    elif arch == "pspnet":
        return smp.PSPNet(encoder_name=backbone, encoder_weights="imagenet", in_channels=3, classes=n_classes)
    else:
        raise ValueError(f"arquitetura não suportada: {arch_name}")

model = create_model(ARCH, BACKBONE, N_CLASSES).to(DEVICE)
print("Modelo:", ARCH, BACKBONE, "| params ≈", sum(p.numel() for p in model.parameters())/1e6, "M")


In [ ]:
#CELDA 26
class DiceLoss_check(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum() + self.eps
        den = probs.sum() + targets.sum() + self.eps
        return 1 - (num/den)

bce  = nn.BCEWithLogitsLoss()
dice = DiceLoss_check()

def bce_plus_dice(logits, targets):
    return bce(logits, targets) + dice(logits, targets)

def _iou_f1_from_counts(tp, fp, fn, eps=1e-7):
    iou = tp/(tp+fp+fn+eps); f1  = (2*tp)/(2*tp+fp+fn+eps); return float(iou), float(f1)

@torch.no_grad()
def eval_imagewise_and_global(model, loader, device="cuda", thr=0.5, logits=True, split_name="VAL"):
    model.eval()
    ious, f1s = [], []
    TP_all = FP_all = FN_all = 0.0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float()
        pb = model(xb)
        if logits: pb = torch.sigmoid(pb)
        pb = (pb >= thr).float()
        B = yb.shape[0]
        for i in range(B):
            y = _to_numpy_bool(yb[i, ...])
            p = _to_numpy_bool(pb[i, ...])
            gt_sum = int(y.sum()); pr_sum = int(p.sum())
            if gt_sum == 0 and pr_sum == 0:
                iou_i, f1_i = 1.0, 1.0
            else:
                tp = int((p & y).sum()); fp = int((p & (~y)).sum()); fn = int(((~p) & y).sum())
                iou_i, f1_i = _iou_f1_from_counts(tp, fp, fn)
                TP_all += tp; FP_all += fp; FN_all += fn
            ious.append(iou_i); f1s.append(f1_i)
    ious = np.array(ious, dtype=np.float32); f1s = np.array(f1s, dtype=np.float32)
    iou_g, f1_g = _iou_f1_from_counts(TP_all, FP_all, FN_all)
    print(f"[{split_name}] [amostra]  F1: {f1s.mean():.6f} ± {f1s.std():.6f} | IoU: {ious.mean():.6f} ± {ious.std():.6f}")
    print(f"[{split_name}] [global]   F1: {f1_g:.6f} | IoU: {iou_g:.6f}")
    return {"sample_mean_iou": float(ious.mean()), "sample_mean_f1": float(f1s.mean()),
            "global_iou": float(iou_g), "global_f1": float(f1_g), "n_images": int(len(ious))}


In [ ]:
##CELDA 27 [16] OPTIM / SCHED / EARLY — Adam + Warmup→Cosine + EarlyStopping
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=LR, weight_decay=WEIGHT_DECAY)
warmup = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=0.0)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience = patience; self.min_delta = min_delta
        self.best = float("inf"); self.count = 0; self.stop = False
    def step(self, val):
        if val < self.best - self.min_delta: self.best = val; self.count = 0
        else:
            self.count += 1
            if self.count >= self.patience: self.stop = True

early = EarlyStopping(patience=PATIENCE_ES, min_delta=0.0)

print("===== CONFIG TREINO =====")
print(f"Loss: BCE+Dice | Optim: Adam(lr={LR}, wd={WEIGHT_DECAY})")
print(f"Scheduler: Warmup({WARMUP_EPOCHS}) + Cosine(T_max={EPOCHS-WARMUP_EPOCHS})")
print(f"EarlyStopping: patience={PATIENCE_ES} | Thr métricas={THRESH_EVAL}")
print("================================")


In [ ]:
##CELDA 28 [TRACE 1] Config + splits + CSV por época
import os, json, glob, csv, platform, torch

# 1) Config del experimento (base)
cfg = dict(
    timestamp=TIMESTAMP,
    seed=int(SEED),
    batch=int(BATCH_SIZE),
    num_aug=int(NUM_AUGMENTED),
    epochs=int(EPOCHS),
    thr=float(THRESH_EVAL),
    drop_last=bool(DROP_LAST),
    model=str(model.__class__.__name__),
    device=str(DEVICE),
    torch=torch.__version__,
    cuda=(torch.version.cuda if torch.cuda.is_available() else None),
    python=platform.python_version(),
)

# --- Añadir trazabilidad de la loss morfológica ANTES de guardar ---
cfg.update(dict(
    morph_on=bool(MORPH_ON),
    morph_bank=str(MORPH_BANK),  # "rect3" | "domain_w2" | "domain_w3" | "dist_auto" | "dist_wX"
    morph_lambda_max=float(LAMBDA_MAX),
    morph_warmup_epochs=int(WARMUP_EPOCHS_MORPH),
    morph_tol_px=int(tol_px),
    bank_dom_w2=[str(k) for k in BANK_DOM_W2],
    bank_dom_w3=[str(k) for k in BANK_DOM_W3],
))
cfg.update(dict(
    use_semi=bool(globals().get("USE_SEMI", False)),
    lambda_u=float(globals().get("LAMBDA_U", 0.0)),
    tau=float(globals().get("TAU", 0.5)),
    semi_start_epoch=int(globals().get("SEMI_START_EPOCH", 0)),
    ema_decay=float(globals().get("EMA_DECAY", 0.0)),
    batch_unlab=int(globals().get("BATCH_SIZE_UNLAB", 0)),
))

# Guardar config con TODO incluido
with open(os.path.join(EXP_DIR, "config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

# 2) Listados de archivos por split (para reproducibilidad)
for split in ["train", "val", "test"]:
    paths = sorted(glob.glob(os.path.join(MSK_ROOT, split, "masks", "*.png")))
    outp  = os.path.join(EXP_DIR, f"{split}_filenames.txt")
    with open(outp, "w") as f:
        f.write("\n".join(os.path.basename(p) for p in paths))

# 3) CSV de métricas por época (supervisadas)
csv_path = os.path.join(EXP_DIR, "train_log.csv")
with open(csv_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch","train_loss","train_dice","train_iou","val_loss"])

def log_epoch_csv(epoch, tr_loss, tr_dice, tr_iou, vl_loss):
    with open(csv_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, tr_loss, tr_dice, tr_iou, vl_loss])

# (Opcional) CSV para métricas de borde si las calculas en validación
bf_path = os.path.join(EXP_DIR, "val_boundary_log.csv")
with open(bf_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch","bf1_tol","assd","hd95"])
def log_val_boundary(epoch, bf1_tol, assd, hd95):
    with open(bf_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, bf1_tol, assd, hd95])

# ===============================
# (NUEVO) CSV detallado por época
# ===============================
csv_path_v2 = os.path.join(EXP_DIR, "train_log_detailed.csv")
with open(csv_path_v2, "w", newline="") as f:
    csv.writer(f).writerow([
            "epoch",
            # TRAIN (componentes)
            "train_loss","train_bce_loss","train_dice_loss","train_morph_loss","train_boundary_loss","train_hd_loss","train_lambda_value",
            # TRAIN (métricas)
            "train_dice_metric","train_iou_metric",
            # VAL (componentes)
            "val_loss","val_bce_loss","val_dice_loss","val_morph_loss","val_boundary_loss","val_hd_loss","val_lambda_value",
            # RAW (train + val)
            "train_morph_raw","train_boundary_raw","train_hd_raw",
            "val_morph_raw","val_boundary_raw","val_hd_raw",
            # SEMI (nuevos)
            "train_unsup_loss","train_lambda_u_t",
            "train_temp_loss","train_lambda_t_t"

        ])

def log_epoch_csv_v2(epoch,
                     # --- TRAIN: componentes ---
                     tr_loss, tr_bce, tr_dice_comp, tr_morph, tr_bound, tr_hd,
                     tr_lambda_value, tr_morph_raw, tr_bound_raw, tr_hd_raw,
                     # --- TRAIN: métricas ---
                     tr_dice_metric, tr_iou_metric,
                     # --- VAL: componentes ---
                     vl_loss, vl_bce, vl_dice, vl_morph, vl_bound, vl_hd,
                     vl_lambda_value, vl_morph_raw, vl_bound_raw, vl_hd_raw,
                      # --- SEMI ---
                     tr_unsup, tr_lambda_u,
                     tr_temp, tr_lambda_t
                     ):
    """
    Escribe 27 columnas (epoch + 26 valores).
    Orden EXACTO para que coincida con tu llamada.
    """
    import csv
    with open(csv_path_v2, "a", newline="") as f:
        csv.writer(f).writerow([
            epoch,
            # TRAIN comp (7)
            tr_loss, tr_bce, tr_dice_comp, tr_morph, tr_bound, tr_hd, tr_lambda_value,
            # TRAIN métricas (2)
            tr_dice_metric, tr_iou_metric,
            # VAL comp (7)
            vl_loss, vl_bce, vl_dice, vl_morph, vl_bound, vl_hd, vl_lambda_value,
            # RAW train (3)
            tr_morph_raw, tr_bound_raw, tr_hd_raw,
            # RAW val (3)
            vl_morph_raw, vl_bound_raw, vl_hd_raw,
            # SEMI (4)
            tr_unsup, tr_lambda_u,
            tr_temp, tr_lambda_t
        ])




In [ ]:
## CELDA 29B [BOUNDARY METRICS HELPERS] — BF1, ASSD, HD95 (por imagen y promedio)

# ===== Imports mínimos y consistentes =====
import numpy as np
import torch
import cv2

# SciPy es preferible para EDT exacta y KDTree; si no está, caemos a fallback
try:
    from scipy.ndimage import distance_transform_edt
    from scipy.spatial import cKDTree
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ==== Constantes globales coherentes (paper) ====
TH_BIN   = 0.5      # umbral de probas (preds.sigmoid() > TH_BIN)
tol_px   = 5        # tolerancia para BF1@r (reporta 3/5/7 en apéndice)
KERNEL_3 = 3        # kernel morfológico para borde (erosión 3x3)
HD_ALPHA = 2.0      # exponentes HD-loss (Karimi)  [no se usan aquí, pero quedan visibles]
HD_BETA  = 2.0

# ===== Helpers NumPy para métricas (post-hoc, CPU) =====
def _to_numpy_bool(x, th=TH_BIN):
    """x: torch.Tensor [B,1,H,W] o [H,W] -> numpy bool [H,W]"""
    if hasattr(x, "detach"):
        x = x.detach().float().cpu().numpy()
    x = np.asarray(x)
    if x.ndim == 4:      # B,1,H,W
        x = x[0, 0]
    elif x.ndim == 3 and x.shape[0] == 1:  # 1,H,W
        x = x[0]
    return (x > th).astype(np.bool_)

def boundary_from_mask_np(m01_u8: np.ndarray) -> np.ndarray:
    """Borde binario = máscara - erosión 3x3 (8-connect)."""
    k = np.ones((KERNEL_3, KERNEL_3), np.uint8)
    er = cv2.erode(m01_u8.astype(np.uint8), k, 1)
    bd = cv2.bitwise_and(m01_u8.astype(np.uint8), cv2.bitwise_not(er))
    return (bd > 0).astype(np.uint8)

def edt_np(bin01_u8: np.ndarray) -> np.ndarray:
    """EDT exacta (preferentemente SciPy; fallback OpenCV)."""
    inv = (bin01_u8 == 0).astype(np.uint8)
    if _HAS_SCIPY:
        return distance_transform_edt(inv)
    # Fallback OpenCV (menos preciso que SciPy pero suficiente para métricas)
    return cv2.distanceTransform(inv, distanceType=cv2.DIST_L2, maskSize=3)

def _surface_points(mask_bool: np.ndarray):
    """Coords (N,2) de píxeles TRUE (y, x)."""
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return np.zeros((0,2), dtype=np.float32)
    return np.stack([ys, xs], axis=1).astype(np.float32)

# ===== Métricas de borde =====
def boundary_f1(pred_mask: np.ndarray, gt_mask: np.ndarray, r: int = tol_px) -> float:
    """BF1 con tolerancia r (en píxeles)."""
    gt_b = boundary_from_mask_np(gt_mask)
    pr_b = boundary_from_mask_np(pred_mask)

    pr_n = int(pr_b.sum()); gt_n = int(gt_b.sum())
    if gt_n == 0 and pr_n == 0:
        return np.nan  # sin borde → ignorar en promedios
    if gt_n == 0 or pr_n == 0:
        return 0.0

    # precisión: pred→gt_borde
    dt_gt = edt_np(gt_b)
    tp_pred = int(((pr_b > 0) & (dt_gt <= r)).sum())
    precision = tp_pred / max(pr_n, 1)

    # recall: gt→pred_borde
    dt_pr = edt_np(pr_b)
    tp_gt = int(((gt_b > 0) & (dt_pr <= r)).sum())
    recall = tp_gt / max(gt_n, 1)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def assd_hd95(pred_mask: np.ndarray, gt_mask: np.ndarray):
    """ASSD y HD95 entre contornos (simétricos) con manejo robusto de casos degenerados."""
    gt_b = boundary_from_mask_np(gt_mask)
    pr_b = boundary_from_mask_np(pred_mask)

    A = _surface_points(gt_b)
    B = _surface_points(pr_b)

    # Penalización finita razonable para el HD cuando solo hay un borde.
    H, W = gt_mask.shape[:2]
    diag = float(np.hypot(H, W))  # ≈ distancia máxima posible en la imagen

    # Casos degenerados
    if len(A) == 0 and len(B) == 0:
        # Sin bordes en ninguno → no hay discrepancia de contorno; ignora en ASSD (NaN) y HD95=0
        return (np.nan, 0.0)
    if len(A) == 0 or len(B) == 0:
        # Solo uno tiene borde → castigo grande pero FINITO, evitando 'inf' en el agregado
        return (np.nan, diag)

    # Distancias punto-a-borde opuesto
    if _HAS_SCIPY:
        treeA = cKDTree(A); treeB = cKDTree(B)
        dA, _ = treeB.query(A, k=1)
        dB, _ = treeA.query(B, k=1)
    else:
        # Fallback O(NM)
        def bf_nn(X, Y):
            if len(Y) == 0:
                # No debería ocurrir porque ya tratamos el caso arriba, pero lo hacemos robusto
                return np.full((len(X),), diag, dtype=np.float32)
            diff = X[:, None, :] - Y[None, :, :]
            dist = np.sqrt((diff**2).sum(axis=2))
            return dist.min(axis=1)
        dA = bf_nn(A, B)
        dB = bf_nn(B, A)

    # Métricas simétricas
    assd = (float(dA.mean()) + float(dB.mean())) / 2.0
    # HD95 por el peor de los dos sentidos
    hd95 = float(max(np.percentile(dA, 95), np.percentile(dB, 95)))
    return (assd, hd95)


# ===== Loop por época (VAL/TEST) =====
@torch.no_grad()
def compute_boundary_metrics_epoch(model, loader, device, thr: float = TH_BIN, r_tol_px: int = tol_px):
    model.eval()
    bf1_list, assd_list, hd95_list = [], [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float()
        logits = model(xb)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= thr).float()   # [B,1,H,W]

        B = preds.shape[0]
        for i in range(B):
            g = _to_numpy_bool(yb[i:i+1], th=thr)   # GT [H,W] bool
            if not g.any():
                continue  # GT vacía → no contribuye a métricas de borde

            p = _to_numpy_bool(preds[i:i+1], th=thr)  # pred [H,W] bool
            bf1 = boundary_f1(p, g, r=r_tol_px)
            a, h = assd_hd95(p, g)
            bf1_list.append(bf1); assd_list.append(a); hd95_list.append(h)

    def _finite_mean(x):
        x = np.asarray(x, dtype=np.float32)
        x = x[np.isfinite(x)]
        return float(x.mean()) if x.size else float("nan")

    return _finite_mean(bf1_list), _finite_mean(assd_list), _finite_mean(hd95_list)


In [ ]:
# === CELDA 29B2 — check métricas de borda (sanity + máscara real) ===
import numpy as np
import os
import cv2

print("\n=== SANITY CHECK MÉTRICAS DE BORDA (SINTÉTICO) ===")

def make_square(h=64, w=64, y0=16, x0=16, size=16):
    """Cuadrado binario simple para tests."""
    m = np.zeros((h, w), np.uint8)
    m[y0:y0+size, x0:x0+size] = 1
    return m

# ---------- PARTE 1: TEST SINTÉTICO (CUADRADO) ----------
gt = make_square()

# 1) Predicción perfecta = GT
pred_equal = gt.copy()

# 2) Predicción desplazada 3 píxeles a la derecha
pred_shift = np.roll(gt, shift=3, axis=1)

# 3) Pred vacía
pred_empty = np.zeros_like(gt)

# 4) Pred llena (todo 1)
pred_full = np.ones_like(gt)

tests = [
    ("pred = GT (perfecto)", pred_equal),
    ("pred = GT desplazada 3px", pred_shift),
    ("pred vacía", pred_empty),
    ("pred llena", pred_full),
]

for name, p in tests:
    bf1 = boundary_f1(p, gt, r=int(tol_px))
    assd, hd95 = assd_hd95(p, gt)
    print(f"\nCaso: {name}")
    print(f"  BF1  = {bf1}")
    print(f"  ASSD = {assd}")
    print(f"  HD95 = {hd95}")

# 5) Comprobar simetría
bf1_ab = boundary_f1(pred_shift, gt, r=int(tol_px))
bf1_ba = boundary_f1(gt, pred_shift, r=int(tol_px))
assd_ab, hd_ab = assd_hd95(pred_shift, gt)
assd_ba, hd_ba = assd_hd95(gt, pred_shift)

print("\n=== CHEQUEO DE SIMETRÍA (SINTÉTICO) ===")
print(f"BF1(pred_shift, gt) = {bf1_ab}")
print(f"BF1(gt, pred_shift) = {bf1_ba}")
print(f"ASSD(pred_shift, gt) = {assd_ab}")
print(f"ASSD(gt, pred_shift) = {assd_ba}")
print(f"HD95(pred_shift, gt) = {hd_ab}")
print(f"HD95(gt, pred_shift) = {hd_ba}")

# ---------- PARTE 2: TEST CON MÁSCARA REAL DEL DATASET ----------
print("\n=== SANITY CHECK MÉTRICAS DE BORDA (MÁSCARA REAL) ===")

# asumimos que MSK_ROOT ya está definido y tiene train/masks
train_masks_dir = os.path.join(MSK_ROOT, "train", "masks")
mask_files = [f for f in os.listdir(train_masks_dir) if f.endswith(".png")]

if len(mask_files) == 0:
    print("No encontré máscaras en:", train_masks_dir)
else:
    mask_path = os.path.join(train_masks_dir, mask_files[0])
    print("Usando máscara:", mask_path)

    gt_real = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_real = (gt_real > 0).astype(np.uint8)

    # pred 1: igual que GT
    pred_equal_real = gt_real.copy()

    # pred 2: suavizada (morfología básica)
    kernel = np.ones((3, 3), np.uint8)
    pred_smooth_real = cv2.morphologyEx(gt_real, cv2.MORPH_OPEN, kernel)

    # pred 3: desplazada 2 px a la derecha
    pred_shift_real = np.roll(gt_real, shift=2, axis=1)

    tests_real = [
        ("pred = GT (perfecto)", pred_equal_real),
        ("pred suavizada (open 3x3)", pred_smooth_real),
        ("pred desplazada 2px", pred_shift_real),
    ]

    for name, p in tests_real:
        bf1 = boundary_f1(p, gt_real, r=int(tol_px))
        assd, hd95 = assd_hd95(p, gt_real)
        print(f"\nCaso REAL: {name}")
        print(f"  BF1  = {bf1}")
        print(f"  ASSD = {assd}")
        print(f"  HD95 = {hd95}")


In [ ]:
## CELDA 29C [BOUNDARY METRICS HELPERS] — BF1, ASSD, HD95 (por imagen y promedio)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms as T

# Parámetros globales por defecto (si no existen)
KERNEL_3 = globals().get("KERNEL_3", 3)
TH_BIN = globals().get("TH_BIN", 0.5)
HD_ALPHA = globals().get("HD_ALPHA", 2.0)
HD_BETA = globals().get("HD_BETA", 2.0)
tol_px = 5
# 🔹 Funções morfológicas
# ======================
def boundary_from_mask_torch(m01_t: torch.Tensor) -> torch.Tensor:
    # m01_t: [B,1,H,W] en {0,1}
    k = torch.ones((1,1,KERNEL_3,KERNEL_3), device=m01_t.device)
    er = (F.conv2d(m01_t, k, padding=KERNEL_3//2) == (KERNEL_3*KERNEL_3))
    return (m01_t.bool() & (~er)).float()

#### CAMBIÉ AQUÍ: EDT aproximada (chamfer) 100% Torch para entrenar en GPU

def edt_chamfer_torch(m01_t: torch.Tensor) -> torch.Tensor:
    # Espera un mapa binario (0/1). Si está vacío, devuelve ceros para evitar INF.
    m = (m01_t > 0).to(m01_t.dtype)
    if torch.all(m == 0):
        return torch.zeros_like(m)

    # Inicializa: 0 en positivos, "grande" en el resto
    H = m.shape[-2]; W = m.shape[-1]
    maxd = float(H + W)  # cota superior razonable
    d = torch.where(m > 0, torch.zeros_like(m), torch.full_like(m, maxd))

    # Dos pasadas de relajación (4-vecinos)
    for _ in range(2):
        d[:, :, :, 1:]  = torch.minimum(d[:, :, :, 1:],  d[:, :, :, :-1] + 1)
        d[:, :, 1:, :]  = torch.minimum(d[:, :, 1:, :],  d[:, :, :-1, :] + 1)
        d[:, :, :, :-1] = torch.minimum(d[:, :, :, :-1], d[:, :, :, 1:]  + 1)
        d[:, :, :-1, :] = torch.minimum(d[:, :, :-1, :], d[:, :, 1:, :]  + 1)

    return d.clamp_(0, maxd)

#### CAMBIÉ AQUÍ: SDF con signo para BoundaryLoss (todo en Torch, sin OpenCV/SciPy)
def signed_distance_from_binary_torch(gt01):
    gt = (gt01 > TH_BIN).float()
    posmask = gt
    negmask = 1 - gt
    if gt01.max().item() == 0.0 or gt.min().item() == 1.0:
      return torch.zeros_like(gt01)

    dist_out = edt_chamfer_torch(negmask)
    dist_in  = edt_chamfer_torch(posmask)
    return dist_out - dist_in

class BoundaryLossCanonical(nn.Module):
    """
    Boundary Loss (Kervadec et al., 2019):
    L = mean( p(x) * φ_G(x) ), con φ_G(x) signed distance (negativo dentro).
    Minimizar empuja p→1 dentro (φ<0) y p→0 fuera (φ>0).
    """
    def __init__(self):
        super().__init__()
    def forward(self, preds, targets):
        p   = torch.sigmoid(preds)
        sdf = signed_distance_from_binary_torch(targets)

        # Nota: NO tomamos |.| ni ReLU; el signo de φ es clave en esta formulación
        return (p * sdf).mean()

class HausdorffDistanceLoss(nn.Module):
    """
    Aproximación diferenciable del Hausdorff (Karimi et al., 2019):
    Penaliza FN/FP ponderados por distancia al borde (de GT y de pred).
    α=β=2 son valores habituales; umbral 0.5 para binarizar la pred al estimar su distancia.
    """
    def __init__(self, alpha=2.0, beta=2.0, th_bin=0.5):
        super().__init__()
        self.alpha = float(alpha)
        self.beta  = float(beta)
        self.th    = float(th_bin)

    def forward(self, preds, targets):
        p  = torch.sigmoid(preds)
        y  = (targets > 0.5).float()

        # Distancia desde GT y desde la PRED binarizada (aprox)
        #  - d_gt penaliza FN (y==1, p~0) según distancia al borde de GT
        #  - d_pr penaliza FP (y==0, p~1) según distancia al borde de la PRED
        d_gt = edt_chamfer_torch(boundary_from_mask_torch(y)).pow(self.alpha)

        pr_bin = (p > self.th).float()
        d_pr = edt_chamfer_torch(boundary_from_mask_torch(pr_bin)).pow(self.beta)

        # Términos tipo L2 en probas, ponderados por distancias (Karimi et al.)
        fn_term = y * (1.0 - p).pow(2) * d_gt      # falsos negativos
        fp_term = (1.0 - y) * p.pow(2) * d_pr      # falsos positivos

        return (fn_term.mean() + fp_term.mean())






def _torch_kernel_from_desc(kdesc):
    """Cria kernel binário (torch.Tensor) a partir de uma descrição."""
    shape, params = kdesc
    h, w = params["h"], params["w"]

    if shape == "rect":
        K = torch.ones((h, w), dtype=torch.float32)
    elif shape == "ellipse":
        y, x = torch.meshgrid(
            torch.linspace(-1, 1, h),
            torch.linspace(-1, 1, w),
            indexing="ij"
        )
        mask = (x**2 + y**2) <= 1.0
        K = mask.float()
    else:
        raise ValueError(f"Forma de kernel desconhecida: {shape}")

    return K


def build_target_kernels_torch(yb: torch.Tensor, bank) -> torch.Tensor:
    """
    Gera alvo morfológico (borda/corona) aplicando (dilate - erode) com vários kernels.
    Retorna T ∈ [0,1] de mesmo shape que yb: [B,1,H,W].
    """
    device = yb.device
    Ts = []

    for kdesc in bank:
        K = _torch_kernel_from_desc(kdesc).to(device)
        kH, kW = K.shape
        pad_h, pad_w = kH // 2, kW // 2

        dil = F.max_pool2d(yb, kernel_size=(kH, kW), stride=1, padding=(pad_h, pad_w))
        ero = -F.max_pool2d(-yb, kernel_size=(kH, kW), stride=1, padding=(pad_h, pad_w))
        T = (dil - ero).clamp(0, 1)
        Ts.append(T)

    return torch.mean(torch.stack(Ts, dim=0), dim=0)

# ======================
# 🔹 Morphological Loss
# ======================

class MorphoLoss(nn.Module):
    """
    Mede a diferença entre as bordas morfológicas (dilatação - erosão)
    das predições e dos alvos.
    """
    def __init__(self, bank=None):
        super().__init__()
        self.bank = bank or [("rect", {"h": 3, "w": 3})]
        self.l1 = nn.L1Loss()

    def forward(self, preds, targets):
        preds_sig = preds.sigmoid()
        T_pred = build_target_kernels_torch(preds_sig, self.bank)
        T_true = build_target_kernels_torch(targets, self.bank)
        return self.l1(T_pred, T_true)
### no usad
def build_target_kernels(yb: torch.Tensor, bank) -> torch.Tensor:
    """
    yb: [B,1,H,W] (0/1). Aplica cada kernel (dilate - erode) y promedia las coronas.
    Retorna T ∈ [0,1].
    """
    y_np = (yb.detach().cpu().numpy() > 0.5).astype(np.uint8)
    B,_,H,W = y_np.shape
    Ts = []
    for kdesc in bank:
        K = _cv_kernel_from_desc(kdesc)
        t_batch = np.zeros((B,1,H,W), np.uint8)
        for i in range(B):
            gt = y_np[i,0]
            dil = cv2.dilate(gt, K, 1)
            ero = cv2.erode (gt, K, 1)
            T  = ((dil - ero) > 0).astype(np.uint8)
            t_batch[i,0] = T
        Ts.append(t_batch)
    T_np = np.mean(np.stack(Ts, axis=0), axis=0)
    return torch.from_numpy(T_np).to(yb.device).float()
#####
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        """
        Focal Loss para tarefas binárias com logits.
        alpha: peso para classe positiva.
        gamma: fator de foco que reduz a perda para exemplos bem classificados.
        reduction: 'mean', 'sum' ou 'none'
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        inputs: logits (antes da sigmoid), forma [B, 1, H, W]
        targets: rótulos binários, forma [B, 1, H, W]
        """
        probas = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")

        p_t = probas * targets + (1 - probas) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_term = (1 - p_t) ** self.gamma

        loss = alpha_t * focal_term * ce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss  # 'none'


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        return 1 - dice


class TextureLoss(nn.Module):
    def __init__(self, device):
        super(TextureLoss, self).__init__()
        # Pega as primeiras camadas da VGG16 (até o relu3_3)
        vgg_pretrained = models.vgg16(pretrained=True).features[:16].to(device)
        # Não treinar a VGG
        for param in vgg_pretrained.parameters():
            param.requires_grad = False
        self.vgg_layers = vgg_pretrained
        self.device = device

        # Normalização da VGG (média e std)
        self.normalize = T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def forward(self, input, target):
        # input e target: [B, C, H, W], valores em [0,1]

        # Se 1 canal, repete para 3 canais (RGB)
        if input.shape[1] == 1:
            input = input.repeat(1, 3, 1, 1)
        if target.shape[1] == 1:
            target = target.repeat(1, 3, 1, 1)

        # Normaliza para média e std da VGG
        #input = self.normalize(input)
        #target = self.normalize(target)

        # Extrai features
        input_features = self.vgg_layers(input)
        target_features = self.vgg_layers(target)

        # Calcula L1 loss entre features
        loss = nn.functional.l1_loss(input_features, target_features)

        return loss


# ======================
# 🔹 Combinação de duas losses
# ======================

def combine(loss_a, loss_b, preds, targets, mode="sum", alpha=0.5):
    """
    Combina duas losses de acordo com o modo especificado.
    """
    la = loss_a(preds, targets)
    lb = loss_b(preds, targets)

    if mode == "sum":
        total = alpha * la + (1 - alpha) * lb
        w_a, w_b = alpha, (1 - alpha)

    elif mode == "dynamic":
        # balanceamento por magnitude das losses
        la_val = la.detach()
        lb_val = lb.detach()

        w_a = 1.0 / (la_val + 1e-6)
        w_b = 1.0 / (lb_val + 1e-6)
        w_sum = w_a + w_b

        w_a = w_a / w_sum
        w_b = w_b / w_sum

        total = w_a * la + w_b * lb

    else:
        raise ValueError(f"Modo '{mode}' inválido. Use 'sum' ou 'dynamic'.")


    return total

# ======================
# 🔹 Seletor principal de losses
# ======================

def get_loss_function(loss_name, device,
                      alpha=0.5, mode="sum",
                      hd_alpha=HD_ALPHA, hd_beta=HD_BETA,
                      w_bce=1.0, w_dice=1.0, w_morph=1.0, w_boundary=1.0, w_hd=1.0,
                      lambda_morph=0.3,
                      lambda_morph_max=0.3,
                      lambda_boundary_max=0.3,   # ← añade esto
                      lambda_hd_max=0.3,         # ← y esto
                      warmup_epochs=15,
                      warmup_start_ep=10):


    """
    Retorna a função de perda de acordo com o nome especificado.
    Suporta combinações entre Dice e MorphoLoss.
    """
    bce = nn.BCEWithLogitsLoss()
    dice = DiceLoss()
    morpho = MorphoLoss(bank=[("rect", {"h":3,"w":3})])
    focal = FocalLoss()
    texture = TextureLoss(device)
    boundary_loss = BoundaryLossCanonical()
    hd_loss       = HausdorffDistanceLoss(alpha=hd_alpha, beta=hd_beta, th_bin=0.5)

    if loss_name == "morpho":
        def loss_fn(preds, targets, epoch=None):
            l = morpho(preds, targets)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": torch.tensor(0., device=preds.device),
                "morph": l.detach(),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": None
            }
            return l, comps
        return loss_fn

    elif loss_name == "dice":
        def loss_fn(preds, targets, epoch=None):
            l = dice(preds, targets)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": l.detach(),
                "morph": torch.tensor(0., device=preds.device),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": None
            }
            return l, comps
        return loss_fn

    elif loss_name == "bce":
        def loss_fn(preds, targets, epoch=None):
            l = bce(preds, targets)
            comps = {
                "bce": l.detach(),
                "dice": torch.tensor(0., device=preds.device),
                "morph": torch.tensor(0., device=preds.device),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": None
            }
            return l, comps
        return loss_fn

    elif loss_name in ("dice_bce", "bce_dice"):
        # BASELINE: 1.0*BCE + 1.0*Dice (equivalente a exp_loss_A)
        def loss_fn(preds, targets, epoch=None):
            l_b  = bce(preds, targets)     # BCE
            l_d  = dice(preds, targets)    # Dice
            total = l_b + l_d

            comps = {
                "bce":   l_b.detach(),     # contribución real
                "dice":  l_d.detach(),     # contribución real
                "morph": torch.tensor(0., device=preds.device),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": None
            }
            return total, comps
        return loss_fn


    elif loss_name == "boundary_loss":
        def loss_fn(preds, targets, epoch=None):
            l = boundary_loss(preds, targets)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": torch.tensor(0., device=preds.device),
                "morph": torch.tensor(0., device=preds.device),
                "boundary": l.detach(),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": None
            }
            return l, comps
        return loss_fn

    elif loss_name == "hd_loss":
        def loss_fn(preds, targets, epoch=None):
            l = hd_loss(preds, targets)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": torch.tensor(0., device=preds.device),
                "morph": torch.tensor(0., device=preds.device),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": l.detach(),
                "lambda_morph": 0.0,
                "alpha_mix": None
            }
            return l, comps
        return loss_fn

    elif loss_name == "dice_morpho":
        # mezcla fija con 'alpha' (dice vs morph)
        def loss_fn(preds, targets, epoch=None):
            l_d = dice(preds, targets)
            l_m = morpho(preds, targets)
            total = combine(dice, morpho, preds, targets, mode=mode, alpha=alpha)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": (alpha * l_d).detach(),
                "morph": ((1.0 - alpha) * l_m).detach(),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": float(alpha)
            }
            return total, comps
        return loss_fn

    elif loss_name == "bce_dice_morpho":
        # BASELINE + MORPH sin warm-up (λ constante):

        def loss_fn(preds, targets, epoch=None):
            l_b = bce(preds, targets)
            l_d = dice(preds, targets)
            l_m = morpho(preds, targets)

            total = l_b + l_d + lambda_morph * l_m

            comps = {
                "bce":   l_b.detach(),
                "dice":  l_d.detach(),
                "morph": (lambda_morph * l_m).detach(),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_value": float(lambda_morph),
                "alpha_mix": None
            }
            return total, comps
        return loss_fn


    elif loss_name == "dice_boundary_loss":
        def loss_fn(preds, targets, epoch=None):
            l_d  = dice(preds, targets)
            l_bo = boundary_loss(preds, targets)
            total = combine(dice, boundary_loss, preds, targets, mode=mode, alpha=alpha)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": (alpha * l_d).detach(),
                "morph": torch.tensor(0., device=preds.device),
                "boundary": ((1.0 - alpha) * l_bo).detach(),
                "hd": torch.tensor(0., device=preds.device),
                "lambda_morph": 0.0,
                "alpha_mix": float(alpha)
            }
            return total, comps
        return loss_fn

    elif loss_name == "dice_hd_loss":
        def loss_fn(preds, targets, epoch=None):
            l_d  = dice(preds, targets)
            l_hd = hd_loss(preds, targets)
            total = combine(dice, hd_loss, preds, targets, mode=mode, alpha=alpha)
            comps = {
                "bce": torch.tensor(0., device=preds.device),
                "dice": (alpha * l_d).detach(),
                "morph": torch.tensor(0., device=preds.device),
                "boundary": torch.tensor(0., device=preds.device),
                "hd": ((1.0 - alpha) * l_hd).detach(),
                "lambda_morph": 0.0,
                "alpha_mix": float(alpha)
            }
            return total, comps
        return loss_fn

    elif loss_name == "bce_dice_morpho_warmup":
        # BASELINE + MORPH con warm-up:
        # total = l_b + l_d + λ_t * l_m

        def _lambda_schedule(epoch):
            # λ_t = 0 si epoch < warmup_start_ep
            # λ_t crece linealmente hasta lambda_morph_max en warmup_epochs
            if epoch is None:
                return 0.0
            if epoch < warmup_start_ep:
                return 0.0
            prog = (epoch - warmup_start_ep + 1) / max(1, warmup_epochs)
            prog = max(0.0, min(1.0, prog))
            return lambda_morph_max * prog

        def loss_fn(preds, targets, epoch=None):
            l_b = bce(preds, targets)       # BCE
            l_d = dice(preds, targets)      # Dice
            l_m = morpho(preds, targets)    # Morph (p.ej., borde)

            lam_t = _lambda_schedule(epoch)  # peso morfológico en esta época

            total = l_b + l_d + lam_t * l_m

            comps = {
                "bce":   l_b.detach(),                 # 1.0 * BCE
                "dice":  l_d.detach(),                 # 1.0 * Dice
                "morph": (lam_t * l_m).detach(),       # λ_t * Morph
                "morph_raw": l_m.detach(),
                "boundary": torch.tensor(0., device=preds.device),
                "boundary_raw": torch.tensor(0., device=preds.device),
                "hd": torch.tensor(0., device=preds.device),
                "hd_raw": torch.tensor(0., device=preds.device),
                "lambda_value": float(lam_t),          # para el log
                "alpha_mix": None
            }
            return total, comps
        return loss_fn

    elif loss_name == "bce_dice_boundary_warmup":
        def _lambda_schedule(epoch):
            if epoch is None: return 0.0
            if epoch < warmup_start_ep: return 0.0
            prog = (epoch - warmup_start_ep + 1) / max(1, warmup_epochs)
            prog = max(0.0, min(1.0, prog))
            return lambda_boundary_max * prog

        def loss_fn(preds, targets, epoch=None):
            l_b = bce(preds, targets)
            l_d = dice(preds, targets)
            l_bd = boundary_loss(preds, targets)  # ← tu función

            lam_t = _lambda_schedule(epoch)
            total = l_b + l_d + lam_t * l_bd

            comps = {
                "bce":   l_b.detach(),
                "dice":  l_d.detach(),
                "morph": torch.tensor(0., device=preds.device),
                "morph_raw": torch.tensor(0., device=preds.device),
                "boundary": (lam_t * l_bd).detach(),
                "boundary_raw": l_bd.detach(),
                "hd": torch.tensor(0., device=preds.device),
                "hd_raw": torch.tensor(0., device=preds.device),
                "lambda_value": float(lam_t),
                "alpha_mix": None
            }
            return total, comps
        return loss_fn

    elif loss_name == "bce_dice_hd_warmup":
        def _lambda_schedule(epoch):
            if epoch is None: return 0.0
            if epoch < warmup_start_ep: return 0.0
            prog = (epoch - warmup_start_ep + 1) / max(1, warmup_epochs)
            prog = max(0.0, min(1.0, prog))
            return lambda_hd_max * prog

        def loss_fn(preds, targets, epoch=None):
            l_b = bce(preds, targets)
            l_d = dice(preds, targets)
            l_hd = hd_loss(preds, targets)        # ← tu función

            lam_t = _lambda_schedule(epoch)
            total = l_b + l_d + lam_t * l_hd

            comps = {
                "bce":   l_b.detach(),
                "dice":  l_d.detach(),
                "morph": torch.tensor(0., device=preds.device),
                "morph_raw": torch.tensor(0., device=preds.device),
                "boundary": torch.tensor(0., device=preds.device),
                "boundary_raw": torch.tensor(0., device=preds.device),
                "hd": (lam_t * l_hd).detach(),
                "hd_raw": l_hd.detach(),
                "lambda_value": float(lam_t),
                "alpha_mix": None
            }
            return total, comps

        return loss_fn

    elif loss_name == "bce_dice_morpho_semi":
        def loss_fn(preds_l, y_l, preds_u_strong, pseudo_u, epoch=None):
            # sup
            l_b = bce(preds_l, y_l)
            l_d = dice(preds_l, y_l)
            # unsup (consistency + morpho nos bordes confiáveis das pseudo)
            lam = _lambda_schedule(epoch)  # similar a warmup
            l_u  = F.binary_cross_entropy_with_logits(
                        preds_u_strong, pseudo_u, reduction="none"
                  )
            # opcional: mascarar por confiança, e combinar com MorphLoss
            l_u = l_u.mean()
            l_m = morpho(preds_u_strong, pseudo_u)  # bordes pseudo
            total = l_b + l_d + lam * (l_u + l_m)
            comps = {...}
            return total, comps
        return loss_fn



        print(f"[Loss config] {active} (w: bce={w_bce}, dice={w_dice}, morph={w_morph}, boundary={w_boundary}, hd={w_hd}),", loss_name)



    else:
        raise ValueError(f"Loss function '{loss_name}' não é reconhecida.")



In [ ]:
## CELDA 29 — Sanity check: 1 batch antes de entrenar (no altera pesos)
model.eval()

def _tofloat(x):
    try:
        return x.item()
    except Exception:
        return float(x)

with torch.no_grad():
    xb, yb = next(iter(train_loader))  # un batch de TRAIN
    print("[Sanity] xb", xb.shape, xb.dtype, "| yb", yb.shape, yb.dtype)

    # Chequeos rápidos de forma y binariedad
    assert xb.dim() == 4 and xb.shape[1] in (1, 3), f"Esperaba imagen BCHW, got {list(xb.shape)}"
    assert yb.dim() == 4 and yb.shape[1] == 1, f"Máscara debe ser [B,1,H,W], got {list(yb.shape)}"
    uniq = torch.unique(yb)
    print("[Sanity] mask unique =", [u.item() for u in uniq])

    xb = xb.to(DEVICE).float()
    yb = yb.to(DEVICE).float()

    logits = model(xb)  # [B,1,H,W]
    assert logits.shape[1] == 1 and logits.shape[2:] == yb.shape[2:], \
        f"Logits y GT deben coincidir en HxW: {list(logits.shape)} vs {list(yb.shape)}"

    # === Loss de sanity ===
    # Si 'criterion' ya está definido (por ej., moviste su creación arriba),
    # úsalo. Si no, calcula una loss rápida BCE+Dice para no romper.
    try:
        _ = criterion  # sólo para verificar existencia
        loss, components = criterion(logits, yb, epoch=None)
        print(f"[Sanity] total_loss={_tofloat(loss):.6f}")
        for k in ["bce","dice","morph","boundary","hd","lambda_value","morph_raw","boundary_raw","hd_raw"]:
            if isinstance(components, dict) and k in components:
                v = components[k]
                v = v.item() if hasattr(v, "item") else float(v)
                print(f"    - {k}: {v:.6f}")
    except NameError:
        # Fallback: BCE + Dice rápidos
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, yb)
        probs = torch.sigmoid(logits)
        smooth = 1e-7
        inter = (probs * yb).sum(dim=(1,2,3))
        denom = probs.sum(dim=(1,2,3)) + yb.sum(dim=(1,2,3))
        dice = 1.0 - (2.0 * inter + smooth) / (denom + smooth)
        dice = dice.mean()
        loss = 0.5 * bce + 0.5 * dice
        print(f"[Sanity] total_loss(BCE+Dice rápido)={_tofloat(loss):.6f}")
        print(f"    - bce:  {bce.item():.6f}")
        print(f"    - dice: {dice.item():.6f}")

    # Rango de probabilidades y métrica rápida
    probs = torch.sigmoid(logits)
    mn, mx = probs.min().item(), probs.max().item()
    print(f"[Sanity] probs in [{mn:.4f}, {mx:.4f}]  thr={THRESH_EVAL}")

    preds = (probs >= THRESH_EVAL).float()
    inter = (preds * yb).sum()
    dice_quick = (2 * inter + 1e-7) / (preds.sum() + yb.sum() + 1e-7)
    print(f"[Sanity] dice_quick={dice_quick.item():.4f}")

model.train()
# === Fin sanity ===


In [ ]:
# ===== DEBUG FINGERPRINT PRE-RUN START =====
_fp_path = os.path.join(EXP_DIR, "debug_fingerprint.json")
_fp_pre  = _fp_collect_pre_legacy()
_fp_save(_fp_pre, None, _fp_path)
print(f"[fingerprint] pre_run saved → {_fp_path}")
# ===== DEBUG FINGERPRINT PRE-RUN END =====

In [ ]:
##CELDA 30 [17] TRAIN — guarda melhor checkpoint por score (0.8 IoU + 0.2 BF1); early stopping por val_loss

from datetime import datetime
import datetime
import time
import torch
import os
import numpy as np
from copy import deepcopy
import torch.nn.functional as F

def _fmt_td(seconds):
    return str(datetime.timedelta(seconds=int(seconds)))

def _cuda_max_mem_mb():
    if torch.cuda.is_available():
        return int(torch.cuda.max_memory_allocated() / (1024**2))
    return 0

best_val_loss = float("inf")
best_ckpt_score = -float("inf")

def _tofloat(x):
    return x.item() if isinstance(x, torch.Tensor) else float(x)

criterion  = get_loss_function(loss_name, DEVICE)


if USE_SEMI and unlabeled_loader is not None:
    teacher_model = deepcopy(model).to(DEVICE)
    for p in teacher_model.parameters():
        p.requires_grad = False
    print("[SEMI] Teacher EMA inicializado.")
else:
    teacher_model = None
    print("[SEMI] Desactivado (sin unlabeled o USE_SEMI=False).")

@torch.no_grad()
def update_ema_teacher(student, teacher, ema_decay):
    if teacher is None:
        return
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data.mul_(ema_decay).add_(ps.data, alpha=(1.0 - ema_decay))


def run_one_epoch(loader, epoch, epochs, train=True, log_every=10,
                  unl_loader=None, teacher=None, temp_loader=None):
    model.train(train)
    tot_loss, tot_dice, tot_iou, n = 0.0, 0.0, 0.0, 0

    tot_bce = tot_dice_comp = tot_morph = tot_bound = tot_hd = 0.0
    tot_lam = 0.0
    tot_morph_raw = tot_bound_raw = tot_hd_raw = 0.0
    tot_unsup = 0.0
    tot_lambda_u = 0.0
    n_unsup = 0
    tot_temp = 0.0
    tot_lambda_t = 0.0
    n_temp = 0

    if train:
        print(f"\n=== Epoch {epoch}/{epochs} ===")

    num_iter = len(loader)
    batch_time = 0.0
    data_time  = 0.0
    end = time.perf_counter()

    # iterador de unlabeled solo en modo train + semi
    use_semi = bool(
        train and USE_SEMI and (unl_loader is not None) and (teacher is not None)
    )
    if use_semi:
        unl_iter = iter(unl_loader)
        teacher.eval()  # teacher siempre en eval
    else:
        unl_iter = None

    # --- Consistencia temporal: configuración ---
    use_temp = bool(train and (temp_loader is not None) and (LAMBDA_T > 0))
    temp_iter = iter(temp_loader) if use_temp else None


    for it, (xb, yb) in enumerate(loader):
        t0 = time.perf_counter()
        data_time += (t0 - end)

        xb = xb.to(DEVICE, non_blocking=True).float()
        yb = yb.to(DEVICE, non_blocking=True).float()
        bs = xb.size(0)
        with torch.set_grad_enabled(train):
            # --------- SUPERVISADO (igual que antes) ----------
            logits = model(xb)
            sup_loss, components = criterion(logits, yb, epoch=epoch)
            loss = sup_loss

            # --------- SEMI-SUPERVISADO (si está activo) ----------
            if use_semi:
                try:
                    xw_u, xs_u = next(unl_iter)
                except StopIteration:
                    unl_iter = iter(unl_loader)
                    xw_u, xs_u = next(unl_iter)

                xw_u = xw_u.to(DEVICE, non_blocking=True).float()
                xs_u = xs_u.to(DEVICE, non_blocking=True).float()
                bs_u = xs_u.size(0)

                with torch.no_grad():
                    logits_teacher = teacher(xw_u)
                    probs_teacher = torch.sigmoid(logits_teacher)

                    # pseudo-label duro
                    pseudo = (probs_teacher >= 0.5).float()
                    # máscara de confianza
                    conf_mask = (probs_teacher >= TAU) | (probs_teacher <= (1.0 - TAU))

                logits_u = model(xs_u)
                unsup_all = F.binary_cross_entropy_with_logits(
                    logits_u, pseudo, reduction="none"
                )

                if conf_mask.any():
                    unsup_loss = (unsup_all * conf_mask.float()).sum() / conf_mask.float().sum()
                else:
                    unsup_loss = torch.zeros((), device=DEVICE)

                # rampa suave de LAMBDA_U después de SEMI_START_EPOCH
                lambda_u_t = 0.0
                if epoch >= SEMI_START_EPOCH:
                    if 'SEMI_WARMUP_EPOCHS' in globals() and SEMI_WARMUP_EPOCHS > 0:
                        # progreso de 0→1 a lo largo de SEMI_WARMUP_EPOCHS
                        w = min(1.0, (epoch - SEMI_START_EPOCH + 1) / SEMI_WARMUP_EPOCHS)
                        lambda_u_t = LAMBDA_U * w
                    else:
                        lambda_u_t = LAMBDA_U

                loss = loss + lambda_u_t * unsup_loss

                # logging claro
                components["unsup"] = float(unsup_loss.detach())
                components["lambda_u_t"] = float(lambda_u_t)

                tot_unsup += float(unsup_loss.detach()) * bs_u
                tot_lambda_u += float(lambda_u_t) * bs_u
                n_unsup += bs_u



                if use_temp and LAMBDA_T > 0.0 and epoch >= TEMP_START_EPOCH:

                    try:
                        xt0, xt1 = next(temp_iter)
                    except StopIteration:
                        temp_iter = iter(temp_loader)
                        xt0, xt1 = next(temp_iter)

                    xt0 = xt0.to(DEVICE, non_blocking=True)
                    xt1 = xt1.to(DEVICE, non_blocking=True)

                    # Predicciones del modelo en dos frames vecinos del mismo video
                    p0 = torch.sigmoid(model(xt0))
                    p1 = torch.sigmoid(model(xt1))

                    # Máscara de confianza temporal (solo píxeles “claros” en p0)
                    conf_mask_t = (p0 >= TAU_TEMP) | (p0 <= (1.0 - TAU_TEMP))

                    # Diferencia cuadrática por píxel
                    temporal_all = (p0 - p1) ** 2  # misma forma que p0, [B,1,H,W]

                    if conf_mask_t.any():
                        temp_loss = (temporal_all * conf_mask_t.float()).sum() / conf_mask_t.float().sum()
                    else:
                        temp_loss = torch.zeros((), device=DEVICE)

                    # Warm-up específico para consistencia temporal
                    lambda_t_t = 0.0
                    if epoch >= TEMP_START_EPOCH:
                        if TEMP_WARMUP_EPOCHS > 0:
                            w_t = min(1.0, (epoch - TEMP_START_EPOCH + 1) / TEMP_WARMUP_EPOCHS)
                            lambda_t_t = LAMBDA_T * w_t
                        else:
                            lambda_t_t = LAMBDA_T

                    loss = loss + lambda_t_t * temp_loss

                    # logging
                    components["temp_consistency"] = float(temp_loss.detach())
                    components["lambda_t_t"] = float(lambda_t_t)
                    bs_t = xt0.size(0)
                    tot_temp += float(temp_loss.detach()) * bs_t
                    tot_lambda_t += float(lambda_t_t) * bs_t
                    n_temp += bs_t



            else:
                components["unsup"] = 0.0
                components["lambda_u_t"] = 0.0


            # --------- BACKWARD & OPTIMIZER ----------
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

                # actualizar teacher EMA solo si usamos semi
                if use_semi:
                    update_ema_teacher(model, teacher, EMA_DECAY)

        # --------- MÉTRICAS SUPERVISADAS (como antes) ----------
        probs = torch.sigmoid(logits)
        preds = (probs >= THRESH_EVAL).float()
        inter = (preds * yb).sum(dim=(1, 2, 3))
        union = preds.sum(dim=(1, 2, 3)) + yb.sum(dim=(1, 2, 3)) - inter
        dice_b = (2 * inter + 1e-7) / (
            preds.sum(dim=(1, 2, 3)) + yb.sum(dim=(1, 2, 3)) + 1e-7
        )
        iou_b = (inter + 1e-7) / (union + 1e-7)


        tot_loss += float(loss.detach()) * bs
        tot_dice += float(dice_b.mean()) * bs
        tot_iou  += float(iou_b.mean()) * bs
        n += bs

        tot_bce       += _tofloat(components.get("bce", 0.0))        * bs
        tot_dice_comp += _tofloat(components.get("dice", 0.0))       * bs
        tot_morph     += _tofloat(components.get("morph", 0.0))      * bs
        tot_bound     += _tofloat(components.get("boundary", 0.0))   * bs
        tot_hd        += _tofloat(components.get("hd", 0.0))         * bs
        tot_morph_raw += _tofloat(components.get("morph_raw", 0.0))  * bs
        tot_bound_raw += _tofloat(components.get("boundary_raw", 0.0)) * bs
        tot_hd_raw    += _tofloat(components.get("hd_raw", 0.0))     * bs
        tot_lam       += _tofloat(components.get("lambda_value", 0.0)) * bs

        # --------- LOGGING ----------
        t1 = time.perf_counter()
        batch_time += (t1 - t0)
        it_done = it + 1
        it_left = num_iter - it_done
        sec_per_it = batch_time / max(it_done, 1)
        eta = sec_per_it * it_left
        lr_cur = optimizer.param_groups[0].get("lr", 0.0)
        avg_so_far = tot_loss / max(1, n)

        if train and (it_done % log_every == 0 or it_done == num_iter):
            print(
                f"Epoch: [{epoch:02d}]\t[{it_done:3d}/{num_iter:3d}]"
                f"\teta: {_fmt_td(eta)}"
                f"\tloss: {float(loss.detach()):.4f} ({avg_so_far:.4f})"
                f"\tlr: {lr_cur:.6f}"
                f"\ttime: {batch_time/it_done:.4f}"
                f"\tdata: {data_time/it_done:.4f}"
                f"\tmax mem: {_cuda_max_mem_mb()}"
            )

        end = time.perf_counter()

    # --------- PROMEDIOS POR EPOCH ----------
    epoch_avg_loss  = tot_loss / max(1, n)
    epoch_dice      = tot_dice / max(1, n)
    epoch_iou       = tot_iou  / max(1, n)
    epoch_bce       = tot_bce / max(1, n)
    epoch_dice_comp = tot_dice_comp / max(1, n)
    epoch_morph     = tot_morph / max(1, n)
    epoch_bound     = tot_bound / max(1, n)
    epoch_hd        = tot_hd / max(1, n)
    epoch_lambda    = tot_lam / max(1, n)
    epoch_morph_raw = tot_morph_raw / max(1, n)
    epoch_bound_raw = tot_bound_raw / max(1, n)
    epoch_hd_raw    = tot_hd_raw / max(1, n)
    epoch_unsup    = (tot_unsup    / max(1, n_unsup)) if n_unsup > 0 else 0.0
    epoch_lambda_u = (tot_lambda_u / max(1, n_unsup)) if n_unsup > 0 else 0.0
    epoch_temp     = (tot_temp     / max(1, n_temp)) if n_temp > 0 else 0.0
    epoch_lambda_t = (tot_lambda_t / max(1, n_temp)) if n_temp > 0 else 0.0

    if train:
        print(
            f"Epoch: [{epoch:02d}] Total time: {_fmt_td(batch_time)} "
            f"({batch_time/max(num_iter,1):.4f} s/it)"
        )
        print(
            f"Averaged stats: loss: {epoch_avg_loss:.4f}\t"
            f"dice: {epoch_dice:.4f}\tiou: {epoch_iou:.4f}\t"
            f"lr: {optimizer.param_groups[0].get('lr',0.0):.6f}"
        )
        print(
            f"semi: unsup={epoch_unsup:.6f}  lambda_u_t={epoch_lambda_u:.4f}  "
            f"temp={epoch_temp:.6f}  lambda_t_t={epoch_lambda_t:.4f}"
        )

        print("USE_SEMI:", USE_SEMI,
      "has_unl:", unl_loader is not None,
      "has_teacher:", teacher is not None,
      "use_semi:", use_semi)


    return (
        epoch_avg_loss,
        epoch_dice, epoch_iou,
        epoch_bce, epoch_dice_comp,
        epoch_morph, epoch_bound, epoch_hd,
        epoch_lambda,
        epoch_morph_raw, epoch_bound_raw, epoch_hd_raw, epoch_unsup, epoch_lambda_u, epoch_temp, epoch_lambda_t)


###############################################################################################
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # TRAIN
    (tr_loss, tr_dice, tr_iou,
     tr_bce, tr_dice_comp,
     tr_morph, tr_bound, tr_hd,
     tr_lambda_value,
     tr_morph_raw, tr_bound_raw, tr_hd_raw, tr_unsup,tr_lambda_u, tr_temp, tr_lambda_t) = run_one_epoch(
        train_loader, epoch, EPOCHS,
        train=True, log_every=10,
        unl_loader=unlabeled_loader,
        teacher=teacher_model,
        temp_loader=temporal_unlab_loader,
    )

    # VAL
    model.eval()
    vl_loss_sum = vl_bce_sum = vl_dice_sum = vl_morph_sum = vl_bound_sum = vl_hd_sum = 0.0
    val_n = 0
    vl_lam_sum = 0.0
    vl_morph_raw_sum = 0.0
    vl_bound_raw_sum = 0.0
    vl_hd_raw_sum = 0.0

    # ### AQUÍ: quita alpha_t (el warm-up ya lo maneja criterion)
    # alpha_t = ALPHA_MAX * max(0.0, min(1.0, (epoch - 1) / max(1, WARMUP_EPOCHS)))

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE, non_blocking=True).float()
            yb = yb.to(DEVICE, non_blocking=True).float()

            logits = model(xb)

            epoch_val = None
            loss, components = criterion(logits, yb, epoch=epoch_val)


            # ### AQUÍ: usa siempre 'components' (s')
            bs = xb.size(0)
            vl_loss_sum  += loss.item() * bs
            vl_bce_sum   += _tofloat(components.get("bce", 0.0))      * bs
            vl_dice_sum  += _tofloat(components.get("dice", 0.0))     * bs
            vl_lam_sum += _tofloat(components.get("lambda_value", 0.0)) * bs


            vl_morph_sum += _tofloat(components.get("morph", 0.0))    * bs
            vl_bound_sum += _tofloat(components.get("boundary", 0.0)) * bs
            vl_hd_sum    += _tofloat(components.get("hd", 0.0))       * bs
            vl_morph_raw_sum += _tofloat(components.get("morph_raw", 0.0)) * bs
            vl_bound_raw_sum += _tofloat(components.get("boundary_raw", 0.0)) * bs
            vl_hd_raw_sum    += _tofloat(components.get("hd_raw", 0.0)) * bs

            val_n        += bs

    vl_loss  = vl_loss_sum  / val_n
    vl_bce   = vl_bce_sum   / val_n
    vl_dice  = vl_dice_sum  / val_n
    vl_morph = vl_morph_sum / val_n
    vl_bound = vl_bound_sum / val_n
    vl_hd    = vl_hd_sum    / val_n
    vl_lambda_value = vl_lam_sum / val_n
    vl_morph_raw = vl_morph_raw_sum / val_n
    vl_bound_raw = vl_bound_raw_sum / val_n
    vl_hd_raw    = vl_hd_raw_sum / val_n

    print(f"Epoch [{epoch}/{EPOCHS}] - Val Loss: {vl_loss:.6f}")

    print(f"[VAL DEBUG] vl_loss={vl_loss:.6f} vl_bce={vl_bce:.6f} vl_dice={vl_dice:.6f} vl_hd={vl_hd:.6f} vl_morph={vl_morph:.6f}")
    print(
        f"[VAL DEBUG] mean(lambda)={vl_lambda_value:.6f} "
        f"mean(hd_raw)={vl_hd_raw:.6f} "
        f"mean(morph_raw)={vl_morph_raw:.6f}"
    )

    log_epoch_csv(epoch, tr_loss, tr_dice, tr_iou, vl_loss)
    if 'log_epoch_csv_v2' in globals():
        log_epoch_csv_v2(
            epoch,
            # train (componentes)
            tr_loss, tr_bce, tr_dice_comp, tr_morph, tr_bound, tr_hd,
            tr_lambda_value, tr_morph_raw, tr_bound_raw, tr_hd_raw,
            # train (métricas)
            tr_dice, tr_iou,
            # val (componentes)
            vl_loss, vl_bce, vl_dice, vl_morph, vl_bound, vl_hd,
            vl_lambda_value, vl_morph_raw, vl_bound_raw, vl_hd_raw,
            # nuevo: término no supervisado medio en train
            tr_unsup, tr_lambda_u,
            tr_temp, tr_lambda_t
        )



    val_metrics  = eval_imagewise_and_global(model, val_loader, device=DEVICE, thr=THRESH_EVAL, logits=True, split_name=f"VAL@{epoch:03d}")
    bf1_mean, assd_mean, hd95_mean = compute_boundary_metrics_epoch(
        model, val_loader, device=DEVICE, thr=THRESH_EVAL, r_tol_px=int(tol_px)
    )
    log_val_boundary(epoch, bf1_mean, assd_mean, hd95_mean)
    print(f"[VAL boundary] BF1@{int(tol_px)}px={bf1_mean:.4f} | ASSD={assd_mean:.3f} | HD95={hd95_mean:.3f}")


    # === NUEVA LÓGICA DE CHECKPOINT: métrica compuesta IoU_global + BF1 ===
    val_iou_global = float(val_metrics["global_iou"])
    bf1_for_score = float(bf1_mean) if np.isfinite(bf1_mean) else 0.0

    ckpt_score = val_iou_global #+ 0.2 * bf1_for_score

    print(f"[VAL ckpt-score] IoU_global={val_iou_global:.4f} | BF1={bf1_for_score:.4f} | "
          f"score=1*IoU={ckpt_score:.6f}")

    if ckpt_score > best_ckpt_score + 1e-6:
        best_ckpt_score = ckpt_score
        best_val_loss = min(best_val_loss, vl_loss)
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ↳ nuevo mejor checkpoint "
              f"(score={best_ckpt_score:.6f}, val_loss={vl_loss:.6f}) guardado en {BEST_PATH}")

    # Scheduler + early stopping (siguen basados en val_loss)
    scheduler.step()
    early.step(-ckpt_score)

    dt = time.time() - t0
    print(f"[{epoch:03d}] {dt:4.1f}s | "
          f"train: loss={tr_loss:.4f} dice={tr_dice:.4f} iou={tr_iou:.4f} | "
          f"val_loss={vl_loss:.4f}")

    if early.stop:
        print(" early stopping was triggered")
        break

print("Mejor score de checkpoint:", best_ckpt_score, "| ckpt:", BEST_PATH)
print("Mejor val_loss observado (solo referencia):", best_val_loss)
print("Loss usada:", loss_name)




In [ ]:
# #CELDA 31 [18] EVAL + EXTRA — avaliação final e inferência visual (prob/bin/overlay + F1/IoU por imagem)

# TIMESTAMP ="2025-09-12-21-18"
#EXP_DIR = os.path.join(ROOT_SAVE_DIR, TIMESTAMP)
#os.makedirs(EXP_DIR, exist_ok=True)
#BEST_PATH = os.path.join(EXP_DIR, "model_best_val_loss.pth")




############### === guardar boundary metrics por imagen (VAL y TEST) con el mismo umbral/tolerancia ===

@torch.no_grad()
def _boundary_metrics_per_image_to_csv(model, loader, split_name, device, thr, tol_px, out_dir):
    rows = []
    model.eval()
    for batch in loader:
        # admite (xb, yb) o (xb, yb, names)
        if isinstance(batch, (list, tuple)):
            if len(batch) == 3:
                xb, yb, _ = batch
            else:
                xb, yb = batch
        else:
            xb, yb = batch

        xb = xb.to(device).float()
        yb = yb.to(device).float()

        logits = model(xb)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= thr).float()

        B = preds.shape[0]
        for i in range(B):
            # convierte a numpy binario (bool/0-1) manteniendo la forma [1,H,W]
            p = (preds[i,0].detach().cpu().numpy() >= 0.5).astype(bool)  # [H,W] bool
            g = (yb[i,0].detach().cpu().numpy()    >= 0.5).astype(bool)  # [H,W] bool

            bf1  = boundary_f1(p, g, r=int(tol_px))
            assd, hd95 = assd_hd95(p, g)

            rows.append(dict(idx=len(rows), bf1=float(bf1),
                             assd=float(assd), hd95=float(hd95)))

    # stats
    bf1_vals  = np.array([r["bf1"]  for r in rows], dtype=np.float32)
    assd_vals = np.array([r["assd"] for r in rows], dtype=np.float32)
    hd95_vals = np.array([r["hd95"] for r in rows], dtype=np.float32)

    # imprime resumen para que coincida con lo mostrado arriba
    print(f"[{split_name} boundary] BF1@{int(tol_px)}px: mean={np.nanmean(bf1_vals):.4f} ± {np.nanstd(bf1_vals):.4f}")
    print(f"[{split_name} boundary] ASSD:              mean={np.nanmean(assd_vals):.3f} ± {np.nanstd(assd_vals):.3f}")
    print(f"[{split_name} boundary] HD95:               mean={np.nanmean(hd95_vals):.3f} ± {np.nanstd(hd95_vals):.3f}")

    # guarda csv
    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"{split_name}_boundary_metrics.csv")
    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["idx","bf1","assd","hd95"])
        w.writeheader(); w.writerows(rows)
    print("guardado:", out_csv)




# avaliação final (carrega melhor checkpoint e avalia VAL/TEST em modo texto)
if os.path.isfile(BEST_PATH):
    model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
    model.to(DEVICE).eval()
    print("\n== Avaliação final ==")

    with torch.no_grad():
        _fp_val_m  = eval_imagewise_and_global(model, val_loader,  device=DEVICE, thr=THRESH_EVAL, logits=True, split_name="VAL")
        _fp_test_m = eval_imagewise_and_global(model, test_loader, device=DEVICE, thr=THRESH_EVAL, logits=True, split_name="TEST")

    # ➕ MÉTRICAS DE BORDE (VAL/TEST)
    print("\n-- Métricas de borde (BF1, ASSD, HD95) --")
    for split_name, loader in [("VAL", val_loader), ("TEST", test_loader)]:
        bf1_mean, assd_mean, hd95_mean = compute_boundary_metrics_epoch(
            model, loader, device=DEVICE, thr=THRESH_EVAL, r_tol_px=int(tol_px)
        )
        print(f"[{split_name}] BF1@r={tol_px}px: {bf1_mean:.6f} | ASSD: {assd_mean:.6f}px | HD95: {hd95_mean:.6f}px")
    # llama para VAL y TEST, reusando las MISMAS variables de tu celda:
    # - model ya cargado con BEST_PATH
    # - DEVICE, THRESH_EVAL, tol_px, EXP_DIR, val_loader, test_loader
    _boundary_metrics_per_image_to_csv(model, val_loader,  "val",  DEVICE, THRESH_EVAL, int(tol_px), EXP_DIR)
    _boundary_metrics_per_image_to_csv(model, test_loader, "test", DEVICE, THRESH_EVAL, int(tol_px), EXP_DIR)

    # ===== DEBUG FINGERPRINT POST-RUN START =====
    _fp_post = _fp_collect_post_legacy(_fp_val_m, _fp_test_m)
    _fp_save(_fp_pre, _fp_post, _fp_path)
    print(f"[fingerprint] post_run saved → {_fp_path}")
    # ===== DEBUG FINGERPRINT POST-RUN END =====

else:
    print("Checkpoint não encontrado:", BEST_PATH)

# pasta para figuras do TEST
OUT_DIR = os.path.join(EXP_DIR, "preds_vis"); os.makedirs(OUT_DIR, exist_ok=True)

def per_image_metrics(pred_bin, gt_bin):
    """pred_bin/gt_bin: arrays booleanos (HxW)"""
    tp = int((pred_bin & gt_bin).sum())
    fp = int((pred_bin & (~gt_bin)).sum())
    fn = int(((~pred_bin) & gt_bin).sum())
    iou = tp / (tp + fp + fn + 1e-7)
    f1  = (2*tp) / (2*tp + fp + fn + 1e-7)
    return f1, iou

# inferência visual no TEST (2x3 painéis para ficar maior)
shown = 0; max_show = 6000
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(DEVICE).float()
        masks  = masks.to(DEVICE).float()
        logits = model(images)
        probs  = torch.sigmoid(logits)  # HxW prob∈[0,1]
        B = images.size(0)

        for i in range(B):
            if shown >= max_show: break

            #  tensores
            img = images[i].cpu().numpy().transpose(1,2,0)                      # HxWx3 [0,1]
            p   = probs[i,0].cpu().numpy()                                       # HxW   [0,1]  (Prob contínua)
            pr  = (p >= THRESH_EVAL).astype(np.uint8)                            # HxW   {0,1}  (Pred binária)
            gt  = (masks[i,0].cpu().numpy() > 0.5).astype(np.uint8)              # HxW   {0,1}

            # métricas por imagem
            f1_i, iou_i = per_image_metrics(pr.astype(bool), gt.astype(bool))
            bf1_i = boundary_f1(pr.astype(bool), gt.astype(bool), r=int(tol_px))

            assd_i, hd95_i = assd_hd95(pr.astype(bool), gt.astype(bool))
            assd_txt = f"{assd_i:.2f}px" if not np.isnan(assd_i) else "—"
            hd95_txt = f"{hd95_i:.2f}px" if not np.isnan(hd95_i) else "—"
            # overlays
            img8 = (np.clip(img,0,1)*255).astype(np.uint8)
            ov_gt = img8.copy();  ov_gt[gt.astype(bool)] = [255, 0, 0]           # GT em vermelho
            ov_pr = img8.copy();  ov_pr[pr.astype(bool)] = [0, 255, 0]           # Pred em verde

            # figura 2x3 ---
            fig, axs = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
            axs = axs.ravel()

            axs[0].imshow(img);                 axs[0].set_title("Imagem");                axs[0].axis("off")
            axs[1].imshow(gt, cmap="gray");     axs[1].set_title("GT (1 = corrosão)");     axs[1].axis("off")

            im2 = axs[2].imshow(p, vmin=0, vmax=1)  # Prob contínua [0,1]
            axs[2].set_title(f"Prob (thr={THRESH_EVAL})"); axs[2].axis("off")
            fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

            axs[3].imshow(pr, cmap="gray");     axs[3].set_title("Pred binária");          axs[3].axis("off")
            axs[4].imshow(cv2.addWeighted(ov_gt, 0.5, ov_pr, 0.5, 0))
            axs[4].set_title("Overlay GT(vermelho)+Pred(verde)"); axs[4].axis("off")

            axs[5].axis("off")
            axs[5].text(
                0.0, 0.9,
                (
                    f"F1: {f1_i:.3f}\n"
                    f"IoU: {iou_i:.3f}\n"
                    f"BF1@{tol_px}px: {bf1_i:.3f}\n"
                    f"ASSD: {assd_txt}   HD95: {hd95_txt}"
                ),
                fontsize=12, family="monospace", va="top"
            )

            plt.suptitle(
                f"F1={f1_i:.3f} | IoU={iou_i:.3f} | BF1@{tol_px}px={bf1_i:.3f} | ASSD={assd_txt} | HD95={hd95_txt}",
                y=1.02
            )


            outp = os.path.join(OUT_DIR, f"test_pred_{shown:03d}.png")
            plt.savefig(outp, dpi=200, bbox_inches="tight")
            plt.show()
            print("Salvo:", outp)

            shown += 1

        if shown >= max_show: break

print(f"Concluído. Arquivos em {OUT_DIR}/")




In [ ]:
#CELDA 34 # [TRACE 2] Guardar predicciones de TEST + métricas por imagen
import numpy as np, csv, cv2, torch, os

out_dir = os.path.join(EXP_DIR, "test_preds")
os.makedirs(out_dir, exist_ok=True)
met_path = os.path.join(EXP_DIR, "test_metrics.csv")
with open(met_path, "w", newline="") as f:
    csv.writer(f).writerow(["index","f1","iou"])

def _per_image_metrics(pr_bin_bool, gt_bin_bool):
    tp = int((pr_bin_bool & gt_bin_bool).sum())
    fp = int((pr_bin_bool & (~gt_bin_bool)).sum())
    fn = int(((~pr_bin_bool) & gt_bin_bool).sum())
    iou = tp / (tp + fp + fn + 1e-7)
    f1  = (2*tp) / (2*tp + fp + fn + 1e-7)
    return f1, iou

# cargar el mejor y evaluar en TEST para volcar artefactos
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model.to(DEVICE).eval()

idx = 0
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(DEVICE).float()
        probs  = torch.sigmoid(model(images))
        preds  = (probs >= THRESH_EVAL).float()   # [B,1,H,W]

        B = images.size(0)
        for i in range(B):
            gt = (masks[i,0].cpu().numpy() > 0.5)         # bool
            pr = (preds[i,0].cpu().numpy()  > 0.5)         # bool

            f1, iou = _per_image_metrics(pr, gt)
            # escribir métrica por imagen
            with open(met_path, "a", newline="") as f:
                csv.writer(f).writerow([idx, f1, iou])
            # guardar máscara 0/255
            cv2.imwrite(os.path.join(out_dir, f"pred_{idx:04d}.png"),
                        (pr.astype(np.uint8) * 255))
            idx += 1

print(f"[TRACE] Guardados: {idx} preds en {out_dir}  |  métricas en {met_path}")


In [ ]:
#CELDA 35
import os
print("BEST_PATH:", BEST_PATH, "| existe:", os.path.isfile(BEST_PATH))
print("EXP_DIR:", EXP_DIR)


In [ ]:
#CELDA 36 — [LOG 2] Resumen final en summary.txt (robusto)
import os, json, csv, numpy as np

summary_path = os.path.join(EXP_DIR, "summary.txt")
cfg_path     = os.path.join(EXP_DIR, "config.json")
train_csv    = os.path.join(EXP_DIR, "train_log.csv")
test_csv     = os.path.join(EXP_DIR, "test_metrics.csv")

# --- leer config si existe ---
cfg = {}
if os.path.isfile(cfg_path):
    with open(cfg_path, "r") as f:
        cfg = json.load(f)

# --- leer train_log y localizar mejor época por val_loss ---
best_row = None
if os.path.isfile(train_csv):
    with open(train_csv, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = [r for r in reader if r and r.get("val_loss","") != ""]
    if rows:
        # convertir tipos con cuidado
        for r in rows:
            try:
                r["epoch"]      = int(r["epoch"])
                r["train_loss"] = float(r["train_loss"])
                r["train_dice"] = float(r["train_dice"])
                r["train_iou"]  = float(r["train_iou"])
                r["val_loss"]   = float(r["val_loss"])
            except Exception:
                pass
        rows = [r for r in rows if isinstance(r.get("val_loss"), float)]
        if rows:
            best_row = sorted(rows, key=lambda r: r["val_loss"])[0]

# --- métricas por imagen en TEST (si existe) ---
f1_mean = f1_std = iou_mean = iou_std = None
if os.path.isfile(test_csv):
    with open(test_csv, "r", newline="") as f:
        reader = csv.reader(f)
        data = list(reader)
    if len(data) >= 2:
        header = [h.strip().lower() for h in data[0]]
        body   = data[1:]
        # intentar por nombre de columna, si no, caer a índices [1]=F1, [2]=IoU
        try:
            f1_idx  = header.index("f1")
            iou_idx = header.index("iou")
        except ValueError:
            f1_idx, iou_idx = 1, 2  # fallback
        f1s, ious = [], []
        for r in body:
            if not r or len(r) <= max(f1_idx, iou_idx):
                continue
            try:
                f1s.append(float(r[f1_idx]))
                ious.append(float(r[iou_idx]))
            except Exception:
                continue
        if f1s:
            f1_arr, iou_arr = np.array(f1s), np.array(ious)
            f1_mean, f1_std = float(f1_arr.mean()), float(f1_arr.std())
            iou_mean, iou_std = float(iou_arr.mean()), float(iou_arr.std())

# --- escribir resumen ---
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("=== Corrosion Segmentation — Run Summary ===\n")
    f.write(f"EXP_DIR: {EXP_DIR}\n")
    f.write(f"BEST_PATH: {BEST_PATH}\n\n")

    if cfg:
        f.write("[Config]\n")
        # destacar campos de la loss morfológica si existen
        keys_first = ["morph_on","morph_bank","morph_lambda_max","morph_warmup_epochs","morph_tol_px"]
        for k in keys_first:
            if k in cfg: f.write(f"- {k}: {cfg[k]}\n")
        # resto de campos
        for k, v in cfg.items():
            if k not in keys_first:
                f.write(f"- {k}: {v}\n")
        f.write("\n")

    if best_row is not None:
        f.write("[Training]\n")
        f.write(f"- Best val_loss: {best_row['val_loss']:.6f} @ epoch {best_row['epoch']}\n")
        f.write(f"- train_loss: {best_row.get('train_loss',''):.6f} | ")
        f.write(f"train_dice: {best_row.get('train_dice',''):.4f} | ")
        f.write(f"train_iou: {best_row.get('train_iou',''):.4f}\n\n")

    if f1_mean is not None:
        f.write("[TEST metrics]\n")
        f.write(f"- Sample-wise  F1:  {f1_mean:.6f} ± {f1_std:.6f}\n")
        f.write(f"- Sample-wise  IoU: {iou_mean:.6f} ± {iou_std:.6f}\n\n")

    # Punteros a artefactos (por si el lector los busca)
    preds_vis = os.path.join(EXP_DIR, "preds_vis")
    preds_bin = os.path.join(EXP_DIR, "test_preds")
    f.write("[Artefactos]\n")
    f.write(f"- preds_vis/:  {preds_vis}\n")
    f.write(f"- test_preds/: {preds_bin}\n")
    f.write(f"- train_log.csv: {train_csv}\n")
    f.write(f"- test_metrics.csv: {test_csv}\n")

print("Resumen guardado en:", summary_path)


In [ ]:
# === CELDA: comparar régua automática (modelo) vs régua manual (Results.csv) ===
import numpy as np
import cv2, math, csv, os
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

#EXP_DIR = "/content/drive/MyDrive/corrosion_runs/efficient/2025-11-19-14-03_bce_dice"

# --- AJUSTA ESTA RUTA A TU DRIVE ---
UNM_ROOT = Path("/content/drive/MyDrive/UNM TalkBank Dysphagia")  # carpeta que tiene 'rotulos'

ROTULOS_DIR = UNM_ROOT / "rotulos"
TARGET_W, TARGET_H = INPUT_SHAPE  # (320,320) en tu código


# ---------- 1) Helpers para leer la régua MANUAL (Results.csv) ----------

def get_corners_from_angle(x: float, y: float, w: float, h: float, angle_degrees: float):
    corners = {
        "top_left":     (x,       y),
        "top_right":    (x + w,   y),
        "bottom_right": (x + w,   y + h),
        "bottom_left":  (x,       y + h),
    }
    q1 = 0 < angle_degrees < 90
    q3 = -180 < angle_degrees < -90
    if q1 or q3:
        p1 = corners["top_right"]
        p2 = corners["bottom_left"]
    else:
        p1 = corners["top_left"]
        p2 = corners["bottom_right"]
    return p1, p2


def extract_manual_line_raw(stem: str):
    """
    stem: por ej. 'v025_f242'
    Lee rotulos/stem/Results.csv y devuelve p1, p2 en coordenadas ORIGINALES (sin pad/resize).
    """
    csv_path = ROTULOS_DIR / stem / "Results.csv"
    if not csv_path.exists():
        return None, None

    df = pd.read_csv(csv_path)
    if df is None or df.empty:
        return None, None

    row = df.iloc[0]
    p1, p2 = get_corners_from_angle(
        row["BX"], row["BY"], row["Width"], row["Height"], row["Angle"]
    )
    return p1, p2  # (x,y) en coords originales


# ---------- 2) Aplicar la MISMA geometría que PREPROCESS a esos puntos ----------

def warp_points_like_preprocess(stem: str, p1_raw, p2_raw):
    """
    Aplica: pad_to_square + resize a (TARGET_W, TARGET_H) sobre las coordenadas.
    Usamos la Mask.tif original para conocer ancho/alto.
    """
    if p1_raw is None or p2_raw is None:
        return None, None

    # Abrimos la máscara original solo para conocer tamaño
    mask_path = ROTULOS_DIR / stem / "Mask.tif"
    if not mask_path.exists():
        print(f"[WARN] Mask.tif no encontrado para {stem}")
        return None, None

    m = Image.open(mask_path)
    w0, h0 = m.size  # ancho, alto original

    side = max(w0, h0)
    pad_x = (side - w0) // 2
    pad_y = (side - h0) // 2

    def _warp_point(pt):
        x, y = pt
        # 1) pad
        x_pad = x + pad_x
        y_pad = y + pad_y
        # 2) resize a (TARGET_W, TARGET_H) manteniendo escala uniforme
        sx = TARGET_W / side
        sy = TARGET_H / side  # en tu caso son iguales
        x_new = x_pad * sx
        y_new = y_pad * sy
        return (float(x_new), float(y_new))

    return _warp_point(p1_raw), _warp_point(p2_raw)


def line_length(p1, p2):
    return float(math.hypot(p2[0] - p1[0], p2[1] - p1[1]))


# ---------- 3) Régua automática desde tu máscara predicha (la que ya usamos antes) ----------

def _corner_inferior_anterior(mask_u8: np.ndarray):
    ys, xs = np.where(mask_u8 > 0)
    if len(xs) == 0:
        return None
    y_min, y_max = ys.min(), ys.max()
    y_thr = y_min + 0.66 * (y_max - y_min)
    idx = np.where(ys >= y_thr)[0]
    if len(idx) == 0:
        idx = np.arange(len(xs))
    xs2, ys2 = xs[idx], ys[idx]
    best = None
    best_score = None
    for x, y in zip(xs2, ys2):
        score = x - 0.5 * y  # pequeño = izquierda y abajo
        if best_score is None or score < best_score:
            best_score = score
            best = (int(x), int(y))
    return best
def c2_c4_from_mask(mask_bin: np.ndarray,
                    min_pixels: int = 80,
                    n_samples: int = 120,
                    slab_half_thickness: float = 2.0,
                    valley_alpha: float = 0.3,
                    n_bands_fallback: int = 4):
    """
    Aproximación de C2–C4 usando PCA + detección de huecos entre vértebras.

    1) PCA sobre todos los píxeles de la máscara -> eje principal.
    2) Perfil 1D f(t): para muchos valores t, contamos cuántos píxeles
       caen en una banda perpendicular al eje (ventana +/- slab_half_thickness).
    3) Buscamos mínimos profundos de f(t) -> huecos entre vértebras.
    4) Definimos segmentos entre cortes (C2 ~ segmento 0, C4 ~ segmento 2).
    5) En cada segmento, usamos _corner_inferior_anterior para elegir el punto
       infero-anterior y calcular la distancia.

    Si no hay huecos suficientes o los segmentos son malos, se hace fallback
    a una versión simple de 4 bandas iguales a lo largo del eje.
    """

    # --- 0) Asegurar máscara como binaria uint8 ---
    mask_u8 = (mask_bin > 0).astype(np.uint8)
    ys, xs = np.where(mask_u8 > 0)
    if len(xs) < min_pixels:
        return None, None, None

    # --- 1) PCA sobre todos los puntos (x,y) ---
    coords = np.stack([xs, ys], axis=1).astype(float)   # (N, 2)
    center = coords.mean(axis=0)
    coords_centered = coords - center

    cov = coords_centered.T @ coords_centered / (coords_centered.shape[0] - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    principal_dir = eigvecs[:, np.argmax(eigvals)]
    norm = np.linalg.norm(principal_dir)
    if norm < 1e-8:
        return None, None, None
    principal_dir = principal_dir / norm

    # Proyección 1D de cada píxel sobre el eje principal
    t = coords_centered @ principal_dir
    t_min, t_max = t.min(), t.max()
    L = t_max - t_min
    if L < 1e-3:
        return None, None, None

    # --- 2) Construir perfil 1D f(t) ---
    # sampleamos a lo largo del eje
    t_grid = np.linspace(t_min, t_max, n_samples)
    f = np.zeros_like(t_grid, dtype=float)

    for i, t0 in enumerate(t_grid):
        sel = np.abs(t - t0) <= slab_half_thickness
        f[i] = sel.sum()

    # si no hay señal, abortar
    if f.max() < 1e-3:
        return None, None, None

    # --- 3) Suavizar el perfil para evitar ruido ---
    kernel = np.array([1., 2., 1.])
    kernel = kernel / kernel.sum()
    f_smooth = np.convolve(f, kernel, mode="same")

    # --- 4) Buscar mínimos profundos (huecos) ---
    valleys_idx = []
    max_f = f_smooth.max()
    thr_valley = max(2.0, valley_alpha * max_f)  # umbral para considerar "hueco"

    for i in range(1, len(f_smooth) - 1):
        if f_smooth[i] < f_smooth[i-1] and f_smooth[i] < f_smooth[i+1] and f_smooth[i] <= thr_valley:
            valleys_idx.append(i)

    # Necesitamos al menos 2 huecos para tener 3 segmentos (C2, C3, C4)
    if len(valleys_idx) >= 2:
        # Ordenamos y nos quedamos con los dos primeros (más craneales)
        valleys_idx = sorted(valleys_idx)
        # t-cuts en el eje
        cuts = [t_grid[i] for i in valleys_idx]
        # definimos límites de segmentos
        bounds = [t_min] + cuts + [t_max]

        # segmentos: [bounds[0],bounds[1]], [bounds[1],bounds[2]], ...
        # queremos segmento 0 (~C2) y segmento 2 (~C4) si existe
        def segment_mask(a, b):
            sel = (t >= a) & (t <= b)
            if sel.sum() < max(10, min_pixels // 6):
                return None
            xs_seg = xs[sel]
            ys_seg = ys[sel]
            m = np.zeros_like(mask_u8, dtype=np.uint8)
            m[ys_seg, xs_seg] = 1
            return m

        # C2: segmento 0
        m_c2 = segment_mask(bounds[0], bounds[1])
        # C4: segmento 2 si existe, sino el último segmento
        if len(bounds) >= 4:
            m_c4 = segment_mask(bounds[2], bounds[3])
        else:
            m_c4 = segment_mask(bounds[-2], bounds[-1])

        if m_c2 is not None and m_c4 is not None:
            p2 = _corner_inferior_anterior(m_c2)
            p4 = _corner_inferior_anterior(m_c4)
            if p2 is not None and p4 is not None:
                (x2, y2), (x4, y4) = p2, p4
                d = float(math.hypot(x4 - x2, y4 - y2))
                return (x2, y2), (x4, y4), d

    # --- 5) Fallback: si no hay huecos claros, usar 4 bandas iguales ---
    # (versión anterior pero más corta)
    def _fallback_bands():
        t_min_f, t_max_f = t_min, t_max
        Lf = t_max_f - t_min_f
        if Lf < 1e-3:
            return None, None, None

        def band_mask(band_idx: int):
            a = t_min_f + Lf * (band_idx / n_bands_fallback)
            b = t_min_f + Lf * ((band_idx + 1) / n_bands_fallback)
            sel = (t >= a) & (t <= b)
            if sel.sum() < max(10, min_pixels // n_bands_fallback):
                return None
            xs_b = xs[sel]
            ys_b = ys[sel]
            m = np.zeros_like(mask_u8, dtype=np.uint8)
            m[ys_b, xs_b] = 1
            return m

        m_c2_b = band_mask(0)
        m_c4_b = band_mask(2 if n_bands_fallback >= 3 else n_bands_fallback - 1)
        if m_c2_b is None or m_c4_b is None:
            return None, None, None

        p2_b = _corner_inferior_anterior(m_c2_b)
        p4_b = _corner_inferior_anterior(m_c4_b)
        if p2_b is None or p4_b is None:
            return None, None, None

        (x2b, y2b), (x4b, y4b) = p2_b, p4_b
        d_b = float(math.hypot(x4b - x2b, y4b - y2b))
        return (x2b, y2b), (x4b, y4b), d_b

    return _fallback_bands()





# ---------- 4) Loop sobre TEST usando las máscaras guardadas (SIN MODELO) ----------

PRED_DIR = os.path.join(EXP_DIR, "test_preds")

d_preds = []
d_gts   = []
names   = []

idx_global = 0
for img_path, _ in test_ds.pairs:
    stem = Path(img_path).stem  # ej. 'v025_f242'

    # 4.1) cargar máscara predicha desde PNG
    pred_path = os.path.join(PRED_DIR, f"pred_{idx_global:04d}.png")
    if not os.path.exists(pred_path):
        print(f"[WARN] no existe {pred_path}, salto")
        idx_global += 1
        continue

    mask_png = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    if mask_png is None:
        print(f"[WARN] no pude leer {pred_path}, salto")
        idx_global += 1
        continue

    mask_bin = (mask_png > 127).astype(np.uint8)

    # régua automática desde la máscara predicha
    p2_pred, p4_pred, d_pred = c2_c4_from_mask(mask_bin)
    if p2_pred is None:
        idx_global += 1
        continue

    # 4.2) régua manual
    p1_raw, p2_raw = extract_manual_line_raw(stem)
    if p1_raw is None:
        idx_global += 1
        continue

    p1_gt, p2_gt = warp_points_like_preprocess(stem, p1_raw, p2_raw)
    if p1_gt is None:
        idx_global += 1
        continue

    d_gt = line_length(p1_gt, p2_gt)

    d_preds.append(d_pred)
    d_gts.append(d_gt)
    names.append(stem)

    idx_global += 1

# ---------- 5) Resultados: estadísticas + gráfico ----------

d_preds = np.array(d_preds)
d_gts   = np.array(d_gts)
abs_err = np.abs(d_preds - d_gts)

print(f"N pares comparados: {len(d_preds)}")
print(f"Distancia GT (px):  mean={d_gts.mean():.2f}, std={d_gts.std():.2f}")
print(f"Distancia pred (px): mean={d_preds.mean():.2f}, std={d_preds.std():.2f}")
print(f"Error absoluto (px): mean={abs_err.mean():.2f}, std={abs_err.std():.2f}, "
      f"max={abs_err.max():.2f}")

plt.figure(figsize=(6,6))
plt.scatter(d_gts, d_preds, alpha=0.6)
mx = max(d_gts.max(), d_preds.max()) * 1.05
plt.plot([0, mx], [0, mx], "r--")
plt.xlabel("C2–C4 manual (px)")
plt.ylabel("C2–C4 automática (px)")
plt.title("Comparación régua manual vs automática")
plt.grid(True)
plt.show()

# --- guardar resultados por imagen en CSV ---
df_cmp = pd.DataFrame({
    "name": names,
    "d_gt_px": d_gts,
    "d_pred_px": d_preds,
    "abs_err_px": abs_err,
    "rel_err_pct": abs_err / np.maximum(d_gts, 1e-6) * 100.0,
})

out_csv = os.path.join(EXP_DIR, "c2c4_distance_comparison.csv")
df_cmp.to_csv(out_csv, index=False)
print("CSV detallado guardado en:", out_csv)

from scipy.stats import pearsonr

r, p = pearsonr(d_gts, d_preds)
ss_res = np.sum((d_preds - d_gts)**2)
ss_tot = np.sum((d_gts - d_gts.mean())**2)
r2 = 1 - ss_res/ss_tot

print(f"Correlación de Pearson r = {r:.3f} (p = {p:.1e})")
print(f"R² (linea identidad)      = {r2:.3f}")

# --- Bland–Altman ---
mean_vals = 0.5 * (d_gts + d_preds)
diff_vals = d_preds - d_gts  # pred − manual

mean_diff = diff_vals.mean()
sd_diff   = diff_vals.std()
loa_low  = mean_diff - 1.96 * sd_diff
loa_high = mean_diff + 1.96 * sd_diff

plt.figure(figsize=(6,5))
plt.scatter(mean_vals, diff_vals, alpha=0.6)
plt.axhline(mean_diff, color="r", linestyle="--", label=f"media = {mean_diff:.2f}")
plt.axhline(loa_low,  color="g", linestyle="--", label=f"LOA- = {loa_low:.2f}")
plt.axhline(loa_high, color="g", linestyle="--", label=f"LOA+ = {loa_high:.2f}")
plt.xlabel("Media (manual, automática) [px]")
plt.ylabel("Diferencia (automática - manual) [px]")
plt.title("Bland–Altman C2–C4 (px)")
plt.legend()
plt.grid(True)
plt.show()

print(f"Bland–Altman: media diff={mean_diff:.2f} px, LOA= [{loa_low:.2f}, {loa_high:.2f}] px")


#MM_PER_PX = 0.3  # ejemplo, inventado
#d_gts_mm   = d_gts * MM_PER_PX
#d_preds_mm = d_preds * MM_PER_PX
#abs_err_mm = abs_err * MM_PER_PX


In [ ]:
# === EXTRA OPCIONAL: visualizar manual vs automática usando las máscaras guardadas ===
df = pd.read_csv(os.path.join(EXP_DIR, "c2c4_distance_comparison.csv"))
df_sorted = df.sort_values("abs_err_px", ascending=False)
n_show = 500  # cuántos casos quieres inspeccionar

PRED_DIR = os.path.join(EXP_DIR, "test_preds")

for idx in range(min(n_show, len(df_sorted))):
    stem = df_sorted.iloc[idx]["name"]
    err  = df_sorted.iloc[idx]["abs_err_px"]
    print(f"\nEjemplo {idx}: {stem}, abs_err = {err:.2f} px")

    # localizar índice global (posición en test_ds.pairs)
    # asumimos que el orden en test_preds es el mismo que en test_ds
    try:
        idx_global = [i for i, (p, _) in enumerate(test_ds.pairs) if Path(p).stem == stem][0]
    except IndexError:
        print(f"[WARN] no encontré {stem} en test_ds.pairs, salto")
        continue

    # 1) cargar máscara predicha desde PNG
    pred_path = os.path.join(PRED_DIR, f"pred_{idx_global:04d}.png")
    if not os.path.exists(pred_path):
        print(f"[WARN] no existe {pred_path}, salto")
        continue

    mask_png = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    if mask_png is None:
        print(f"[WARN] no pude leer {pred_path}, salto")
        continue

    mask_bin = (mask_png > 127).astype(np.uint8)

    # 2) régua automática desde esa máscara
    p2_pred, p4_pred, d_pred = c2_c4_from_mask(mask_bin)
    if p2_pred is None:
        print("[WARN] no se pudo estimar régua automática, salto")
        continue

    # 3) régua manual a INPUT_SHAPE
    p1_raw, p2_raw = extract_manual_line_raw(stem)
    if p1_raw is None:
        print("[WARN] no se pudo leer Results.csv, salto")
        continue

    p1_gt, p2_gt = warp_points_like_preprocess(stem, p1_raw, p2_raw)
    if p1_gt is None:
        print("[WARN] no se pudo warpear régua manual, salto")
        continue

    # 4) dibujar todo sobre la máscara como fondo
    vis = cv2.cvtColor(mask_png, cv2.COLOR_GRAY2RGB)

    # línea manual en rojo
    cv2.line(
        vis,
        (int(p1_gt[0]), int(p1_gt[1])),
        (int(p2_gt[0]), int(p2_gt[1])),
        (255, 0, 0), 2
    )

    # línea automática en amarillo
    cv2.line(vis, p2_pred, p4_pred, (255, 255, 0), 2)

    plt.figure(figsize=(5,5))
    plt.imshow(vis)
    plt.title(f"{stem}  |  err = {err:.1f} px  |  d_pred={d_pred:.1f} px")
    plt.axis("off")
    plt.show()
